<a href="https://colab.research.google.com/github/KhaledTachyonAlgernon/GrandNexus/blob/main/GrandNexus_Mini_Enhanced_Version_Colab_Optimis%C3%A9e_08102025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# GrandNexus Mini Enhanced - Version Colab Optimisée
# Version progressive vers GrandNexus 6.4 complète
# ============================================================

import asyncio
import logging
import time
import random
import pickle
import os
import json
import textwrap
from collections import defaultdict, OrderedDict, deque
from typing import Any, Callable, Dict, List, Optional, Union, Set
import numpy as np
from dataclasses import dataclass, field
import threading
from abc import ABC, abstractmethod

# Configuration logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("GrandNexusMini")

# ============================================================
# ARCHITECTURE DE BASE OPTIMISÉE POUR COLAB
# ============================================================

class Message:
    """Message entre modules avec métadonnées enrichies"""
    def __init__(self, source: str, target: str, msg_type: str, content: Any,
                 priority: int = 0, semantic_tags: List[str] = None):
        self.source = source
        self.target = target
        self.msg_type = msg_type
        self.content = content
        self.priority = priority
        self.semantic_tags = semantic_tags or []
        self.timestamp = time.time()
        self.id = f"{source}_{target}_{int(time.time()*1000)}"

    def __repr__(self):
        return f"Message({self.source}→{self.target}:{self.msg_type})"

class AsyncMessageBus:
    """Bus de messages optimisé avec routage sémantique"""
    def __init__(self):
        self._subscribers: Dict[str, List[Callable]] = defaultdict(list)
        self.history: List[Message] = []
        self._queue: asyncio.Queue = asyncio.Queue(maxsize=1000)  # Limite pour Colab
        self._worker_task = None
        self._semantic_routes: Dict[str, Set[str]] = defaultdict(set)

    async def start(self):
        if self._worker_task is None:
            self._worker_task = asyncio.create_task(self._worker())
            logger.info("Enhanced message bus started")

    async def stop(self):
        if self._worker_task:
            self._worker_task.cancel()
            try:
                await self._worker_task
            except asyncio.CancelledError:
                pass
            logger.info("Message bus stopped")

    async def _worker(self):
        """Worker avec priorité et routage sémantique"""
        while True:
            try:
                message = await self._queue.get()
                self.history.append(message)

                # Routage sémantique
                relevant_subscribers = self._get_semantic_subscribers(message)

                # Dispatch avec gestion d'erreur
                for callback in relevant_subscribers:
                    try:
                        if asyncio.iscoroutinefunction(callback):
                            await callback(message)
                        else:
                            callback(message)
                    except Exception as e:
                        logger.error(f"Error in message callback: {e}")

            except asyncio.CancelledError:
                break
            except Exception as e:
                logger.error(f"Message bus worker error: {e}")

    def _get_semantic_subscribers(self, message: Message) -> List[Callable]:
        """Trouve les abonnés pertinents basé sur le contenu sémantique"""
        subscribers = self._subscribers.get(message.msg_type, [])

        # Ajoute les abonnés des tags sémantiques
        for tag in message.semantic_tags:
            subscribers.extend(self._subscribers.get(f"semantic:{tag}", []))

        return list(set(subscribers))  # Déduplique

    async def publish(self, message: Message):
        """Publication avec buffer overflow protection"""
        try:
            await self._queue.put(message)
        except asyncio.QueueFull:
            logger.warning("Message queue full, dropping oldest message")
            # En cas de surcharge, on pourrait implémenter une stratégie plus sophistiquée

    def subscribe(self, msg_type: str, callback: Callable):
        """Abonnement avec support des tags sémantiques"""
        self._subscribers[msg_type].append(callback)

    def subscribe_semantic(self, tag: str, callback: Callable):
        """Abonnement basé sur les tags sémantiques"""
        self._subscribers[f"semantic:{tag}"].append(callback)

# ============================================================
# MÉMOIRE HIÉRARCHIQUE AVEC COMPRESSION SÉMANTIQUE
# ============================================================

@dataclass
class MemoryItem:
    """Item de mémoire avec métadonnées enrichies"""
    key: str
    value: Any
    timestamp: float
    access_count: int = 0
    importance: float = 0.5
    provenance: str = "unknown"
    confidence: float = 0.5
    semantic_tags: List[str] = field(default_factory=list)
    compression_level: int = 0  # 0=raw, 1=compressed, 2=abstract
    ttl: Optional[float] = None

class EnhancedHierarchicalMemory:
    """Mémoire hiérarchique avec compression sémantique progressive"""

    def __init__(self, bus: AsyncMessageBus, max_hot_cache: int = 100,
                 max_compressed: int = 500, max_abstract: int = 200):
        self.bus = bus
        self.max_hot_cache = max_hot_cache
        self.max_compressed = max_compressed
        self.max_abstract = max_abstract

        # Trois niveaux de mémoire
        self.hot_cache = OrderedDict()      # Accès rapide, données récentes
        self.compressed_store = {}          # Données compressées sémantiquement
        self.abstract_store = {}            # Concepts et patterns abstraits

        # Métriques et indexes
        self.access_patterns = defaultdict(int)
        self.semantic_index = defaultdict(set)  # tag -> set of keys
        self.concept_graph = defaultdict(set)   # concept relationships

        # Bus subscriptions
        bus.subscribe("memory_request", self._handle_memory_request)
        bus.subscribe_semantic("learning", self._handle_learning_event)

    async def store(self, key: str, value: Any, **metadata) -> None:
        """Stockage intelligent avec classification automatique"""

        # Crée l'item avec métadonnées
        item = MemoryItem(
            key=key,
            value=value,
            timestamp=time.time(),
            **metadata
        )

        # Classification automatique par importance/type
        if item.importance > 0.8 or "critical" in item.semantic_tags:
            await self._store_hot(item)
        elif item.importance > 0.4 or any(tag in ["concept", "pattern"] for tag in item.semantic_tags):
            await self._store_compressed(item)
        else:
            await self._store_abstract(item)

        # Mise à jour des indexes sémantiques
        self._update_semantic_index(item)

        # Éviction si nécessaire
        await self._evict_if_needed()

    async def _store_hot(self, item: MemoryItem):
        """Stockage dans le cache chaud"""
        self.hot_cache[item.key] = item
        self.hot_cache.move_to_end(item.key)  # LRU

    async def _store_compressed(self, item: MemoryItem):
        """Stockage avec compression sémantique"""
        # Compression basique: extraction des concepts clés
        compressed_value = self._compress_semantically(item.value)
        item.value = compressed_value
        item.compression_level = 1
        self.compressed_store[item.key] = item

    async def _store_abstract(self, item: MemoryItem):
        """Stockage des abstractions de haut niveau"""
        # Abstraction: garde seulement les métadonnées et les relations
        abstract_value = {
            "type": type(item.value).__name__,
            "summary": str(item.value)[:100] + "..." if len(str(item.value)) > 100 else str(item.value),
            "concepts": self._extract_concepts(item.value)
        }
        item.value = abstract_value
        item.compression_level = 2
        self.abstract_store[item.key] = item

    def _compress_semantically(self, value: Any) -> Any:
        """Compression sémantique basique"""
        if isinstance(value, str):
            # Garde les mots-clés importants
            words = value.split()
            keywords = [w for w in words if len(w) > 3 and w.lower() not in
                       ['the', 'and', 'that', 'this', 'with', 'from', 'they', 'have', 'been']]
            return " ".join(keywords[:10])  # Top 10 keywords
        elif isinstance(value, dict):
            # Garde les clés importantes
            important_keys = ['type', 'name', 'id', 'content', 'value', 'result']
            return {k: v for k, v in value.items() if k in important_keys}
        return value

    def _extract_concepts(self, value: Any) -> List[str]:
        """Extraction de concepts de base"""
        if isinstance(value, str):
            # Mots de plus de 4 caractères comme concepts
            words = value.lower().split()
            return [w for w in words if len(w) > 4][:5]
        elif isinstance(value, dict) and 'type' in value:
            return [value['type']]
        return []

    def _update_semantic_index(self, item: MemoryItem):
        """Met à jour l'index sémantique"""
        for tag in item.semantic_tags:
            self.semantic_index[tag].add(item.key)

        # Index les concepts extraits
        concepts = self._extract_concepts(item.value)
        for concept in concepts:
            self.semantic_index[f"concept:{concept}"].add(item.key)

    async def retrieve(self, key: str) -> Optional[MemoryItem]:
        """Récupération intelligente multi-niveaux"""

        # Recherche dans le cache chaud
        if key in self.hot_cache:
            item = self.hot_cache[key]
            item.access_count += 1
            self.hot_cache.move_to_end(key)  # LRU update
            return item

        # Recherche dans le stockage compressé
        if key in self.compressed_store:
            item = self.compressed_store[key]
            item.access_count += 1
            # Promotion vers hot cache si très accédé
            if item.access_count > 5:
                await self._promote_to_hot(item)
            return item

        # Recherche dans l'abstract store
        if key in self.abstract_store:
            item = self.abstract_store[key]
            item.access_count += 1
            return item

        return None

    async def _promote_to_hot(self, item: MemoryItem):
        """Promotion d'un item vers le cache chaud"""
        # Retire de compressed store
        if item.key in self.compressed_store:
            del self.compressed_store[item.key]

        # Ajoute au hot cache
        self.hot_cache[item.key] = item
        self.hot_cache.move_to_end(item.key)

    async def search_semantic(self, tags: List[str], limit: int = 10) -> List[MemoryItem]:
        """Recherche sémantique multi-tags"""
        candidate_keys = set()

        for tag in tags:
            candidate_keys.update(self.semantic_index.get(tag, set()))
            candidate_keys.update(self.semantic_index.get(f"concept:{tag}", set()))

        # Récupère les items et les trie par pertinence
        items = []
        for key in candidate_keys:
            item = await self.retrieve(key)
            if item:
                # Score basé sur le nombre de tags matchés et l'importance
                score = len(set(tags) & set(item.semantic_tags)) * item.importance
                items.append((score, item))

        # Trie par score décroissant
        items.sort(key=lambda x: x[0], reverse=True)
        return [item for score, item in items[:limit]]

    async def _evict_if_needed(self):
        """Éviction intelligente des caches"""

        # Éviction du hot cache
        while len(self.hot_cache) > self.max_hot_cache:
            # Évicte le LRU (Least Recently Used)
            old_key, old_item = self.hot_cache.popitem(last=False)

            # Rétrograde vers compressed si important
            if old_item.importance > 0.3:
                await self._store_compressed(old_item)

        # Éviction du compressed store
        while len(self.compressed_store) > self.max_compressed:
            # Évicte les moins importants
            items = list(self.compressed_store.items())
            items.sort(key=lambda x: x[1].importance)
            old_key, old_item = items[0]
            del self.compressed_store[old_key]

            # Rétrograde vers abstract si pertinent
            if len(old_item.semantic_tags) > 0:
                await self._store_abstract(old_item)

    async def _handle_memory_request(self, message: Message):
        """Gestionnaire des requêtes mémoire via bus"""
        if message.content.get('action') == 'retrieve':
            key = message.content.get('key')
            item = await self.retrieve(key)
            response = Message(
                source="memory",
                target=message.source,
                msg_type="memory_response",
                content={'key': key, 'item': item}
            )
            await self.bus.publish(response)

    async def _handle_learning_event(self, message: Message):
        """Traite les événements d'apprentissage pour ajuster l'importance"""
        if 'pattern' in message.content:
            pattern = message.content['pattern']
            # Augmente l'importance des items liés à ce pattern
            for key in self.semantic_index.get(f"concept:{pattern}", set()):
                item = await self.retrieve(key)
                if item:
                    item.importance = min(1.0, item.importance + 0.1)

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques de la mémoire"""
        return {
            "hot_cache_size": len(self.hot_cache),
            "compressed_size": len(self.compressed_store),
            "abstract_size": len(self.abstract_store),
            "semantic_tags_count": len(self.semantic_index),
            "total_concepts": sum(1 for k in self.semantic_index.keys() if k.startswith("concept:")),
            "average_access_patterns": np.mean(list(self.access_patterns.values())) if self.access_patterns else 0
        }

# ============================================================
# GRAPHE CONCEPTUEL AVEC IMAGINATION ET ÉMERGENCE
# ============================================================

class ConceptualNode:
    """Nœud conceptuel enrichi"""
    def __init__(self, name: str, data: Dict[str, Any]):
        self.name = name
        self.data = data
        self.creation_time = time.time()
        self.access_count = 0
        self.confidence = data.get('confidence', 0.5)
        self.plane = data.get('plane', 'real')  # real, imagination, abstract
        self.connections = defaultdict(float)  # node_name -> strength
        self.evolution_history = []

class EmergentConceptualGraph:
    """Graphe conceptuel avec émergence et imagination"""

    def __init__(self, bus: AsyncMessageBus):
        self.bus = bus
        self.nodes: Dict[str, ConceptualNode] = {}
        self.imagination_nodes: Dict[str, ConceptualNode] = {}  # Concepts hypothétiques
        self.emergence_threshold = 0.7
        self.imagination_decay = 0.95

        # Patterns émergents détectés
        self.detected_patterns = {}
        self.pattern_evolution = []

        # Subscriptions
        bus.subscribe("graph_query", self._handle_graph_query)
        bus.subscribe("concept_creation", self._handle_concept_creation)
        bus.subscribe_semantic("emergence", self._handle_emergence_event)

    async def add_concept(self, name: str, data: Dict[str, Any], plane: str = "real") -> ConceptualNode:
        """Ajoute un concept avec classification de plan"""
        data['plane'] = plane
        node = ConceptualNode(name, data)

        if plane == "imagination":
            self.imagination_nodes[name] = node
        else:
            self.nodes[name] = node

        # Détecte les connexions potentielles
        await self._detect_auto_connections(node)

        # Notifie la création
        message = Message(
            source="graph",
            target="*",
            msg_type="concept_created",
            content={'name': name, 'plane': plane, 'data': data},
            semantic_tags=['concept', 'creation', plane]
        )
        await self.bus.publish(message)

        return node

    async def _detect_auto_connections(self, new_node: ConceptualNode):
        """Détecte automatiquement les connexions avec les concepts existants"""

        new_concepts = self._extract_concepts_from_node(new_node)

        # Compare avec les nœuds existants
        target_nodes = self.nodes if new_node.plane != "imagination" else {**self.nodes, **self.imagination_nodes}

        for existing_name, existing_node in target_nodes.items():
            if existing_name == new_node.name:
                continue

            existing_concepts = self._extract_concepts_from_node(existing_node)

            # Calcule la similarité conceptuelle
            similarity = self._calculate_conceptual_similarity(new_concepts, existing_concepts)

            if similarity > 0.3:  # Seuil de connexion
                await self.connect_concepts(new_node.name, existing_name, similarity)

    def _extract_concepts_from_node(self, node: ConceptualNode) -> Set[str]:
        """Extrait les concepts d'un nœud pour comparaison"""
        concepts = set()

        # Concepts du nom du nœud
        concepts.add(node.name.lower())

        # Concepts des données
        for key, value in node.data.items():
            if isinstance(value, str):
                concepts.update(value.lower().split())
            elif key in ['type', 'category', 'class']:
                concepts.add(str(value).lower())

        # Filtre les mots trop courts
        return {c for c in concepts if len(c) > 2}

    def _calculate_conceptual_similarity(self, concepts1: Set[str], concepts2: Set[str]) -> float:
        """Calcule la similarité conceptuelle entre deux ensembles de concepts"""
        if not concepts1 or not concepts2:
            return 0.0

        intersection = len(concepts1 & concepts2)
        union = len(concepts1 | concepts2)

        return intersection / union if union > 0 else 0.0

    async def connect_concepts(self, name1: str, name2: str, strength: float = 0.5):
        """Connecte deux concepts avec une force donnée"""

        # Trouve les nœuds dans tous les plans
        node1 = self.nodes.get(name1) or self.imagination_nodes.get(name1)
        node2 = self.nodes.get(name2) or self.imagination_nodes.get(name2)

        if node1 and node2:
            node1.connections[name2] = strength
            node2.connections[name1] = strength

            # Détecte l'émergence potentielle
            await self._check_emergence(node1, node2, strength)

    async def _check_emergence(self, node1: ConceptualNode, node2: ConceptualNode, connection_strength: float):
        """Vérifie si la connexion génère une émergence"""

        # Critères d'émergence
        emergence_score = (
            connection_strength * 0.4 +
            (node1.confidence + node2.confidence) / 2 * 0.3 +
            min(node1.access_count, node2.access_count) / 100 * 0.3
        )

        if emergence_score > self.emergence_threshold:
            await self._trigger_emergence(node1, node2, emergence_score)

    async def _trigger_emergence(self, node1: ConceptualNode, node2: ConceptualNode, score: float):
        """Déclenche un événement d'émergence"""

        # Crée un nouveau concept émergent
        emergent_name = f"emergent_{node1.name}_{node2.name}_{int(time.time())}"
        emergent_data = {
            'type': 'emergent_concept',
            'parents': [node1.name, node2.name],
            'emergence_score': score,
            'plane': 'emergent'
        }

        emergent_node = await self.add_concept(emergent_name, emergent_data, plane="emergent")

        # Connecte aux parents
        await self.connect_concepts(emergent_name, node1.name, score)
        await self.connect_concepts(emergent_name, node2.name, score)

        # Log et notifie
        logger.info(f"🌟 EMERGENCE: {emergent_name} from {node1.name} + {node2.name} (score: {score:.2f})")

        message = Message(
            source="graph",
            target="*",
            msg_type="emergence_event",
            content={
                'emergent_concept': emergent_name,
                'parents': [node1.name, node2.name],
                'score': score
            },
            semantic_tags=['emergence', 'creation', 'pattern']
        )
        await self.bus.publish(message)

    async def promote_imagination(self, threshold: float = 0.6) -> List[str]:
        """Promeut les concepts d'imagination vers la réalité"""
        promoted = []

        for name, node in list(self.imagination_nodes.items()):
            # Critères de promotion
            if (node.confidence > threshold and
                node.access_count > 3 and
                len(node.connections) > 1):

                # Promeut vers la réalité
                node.plane = "promoted"
                self.nodes[name] = node
                del self.imagination_nodes[name]
                promoted.append(name)

                logger.info(f"✨ PROMOTED: {name} from imagination to reality")

        return promoted

    async def decay_imagination(self):
        """Fait décroître la confiance des concepts d'imagination"""
        to_remove = []

        for name, node in self.imagination_nodes.items():
            node.confidence *= self.imagination_decay

            # Supprime si trop faible
            if node.confidence < 0.1:
                to_remove.append(name)

        for name in to_remove:
            del self.imagination_nodes[name]

    async def detect_patterns(self) -> Dict[str, Any]:
        """Détecte les patterns émergents dans le graphe"""
        patterns = {}

        # Pattern 1: Clusters de concepts fortement connectés
        clusters = self._detect_clusters()
        if clusters:
            patterns['clusters'] = clusters

        # Pattern 2: Chaînes conceptuelles
        chains = self._detect_conceptual_chains()
        if chains:
            patterns['chains'] = chains

        # Pattern 3: Concepts centraux (hubs)
        hubs = self._detect_concept_hubs()
        if hubs:
            patterns['hubs'] = hubs

        # Stocke les patterns détectés
        self.detected_patterns = patterns
        self.pattern_evolution.append({
            'timestamp': time.time(),
            'patterns': patterns
        })

        return patterns

    def _detect_clusters(self) -> List[List[str]]:
        """Détecte les clusters de concepts connectés"""
        clusters = []
        visited = set()

        for node_name in self.nodes:
            if node_name not in visited:
                cluster = self._explore_cluster(node_name, visited, threshold=0.4)
                if len(cluster) > 2:  # Clusters significatifs seulement
                    clusters.append(cluster)

        return clusters

    def _explore_cluster(self, start_node: str, visited: Set[str], threshold: float) -> List[str]:
        """Explore un cluster par DFS"""
        cluster = []
        stack = [start_node]

        while stack:
            current = stack.pop()
            if current in visited:
                continue

            visited.add(current)
            cluster.append(current)

            # Ajoute les voisins fortement connectés
            if current in self.nodes:
                for neighbor, strength in self.nodes[current].connections.items():
                    if strength >= threshold and neighbor not in visited:
                        stack.append(neighbor)

        return cluster

    def _detect_conceptual_chains(self) -> List[List[str]]:
        """Détecte les chaînes conceptuelles (A->B->C->...)"""
        chains = []

        for start_node in self.nodes:
            chain = self._follow_chain(start_node, max_length=5)
            if len(chain) > 2:
                chains.append(chain)

        return chains

    def _follow_chain(self, start_node: str, max_length: int) -> List[str]:
        """Suit une chaîne conceptuelle"""
        chain = [start_node]
        current = start_node
        visited = {start_node}

        for _ in range(max_length - 1):
            if current not in self.nodes:
                break

            # Trouve la connexion la plus forte non visitée
            best_next = None
            best_strength = 0

            for neighbor, strength in self.nodes[current].connections.items():
                if neighbor not in visited and strength > best_strength:
                    best_next = neighbor
                    best_strength = strength

            if best_next and best_strength > 0.3:
                chain.append(best_next)
                visited.add(best_next)
                current = best_next
            else:
                break

        return chain

    def _detect_concept_hubs(self) -> List[Dict[str, Any]]:
        """Détecte les concepts centraux (hubs)"""
        hubs = []

        for name, node in self.nodes.items():
            connection_count = len(node.connections)
            total_strength = sum(node.connections.values())

            if connection_count > 3 and total_strength > 2.0:
                hubs.append({
                    'name': name,
                    'connections': connection_count,
                    'total_strength': total_strength,
                    'hub_score': connection_count * total_strength
                })

        # Trie par score de hub
        hubs.sort(key=lambda x: x['hub_score'], reverse=True)
        return hubs[:5]  # Top 5 hubs

    async def _handle_graph_query(self, message: Message):
        """Gestionnaire des requêtes de graphe"""
        query_type = message.content.get('type')

        if query_type == 'find_concept':
            name = message.content.get('name')
            node = self.nodes.get(name) or self.imagination_nodes.get(name)
            response_content = {'found': node is not None, 'node': node}

        elif query_type == 'get_connections':
            name = message.content.get('name')
            node = self.nodes.get(name) or self.imagination_nodes.get(name)
            connections = dict(node.connections) if node else {}
            response_content = {'connections': connections}

        elif query_type == 'detect_patterns':
            patterns = await self.detect_patterns()
            response_content = {'patterns': patterns}

        else:
            response_content = {'error': 'Unknown query type'}

        response = Message(
            source="graph",
            target=message.source,
            msg_type="graph_response",
            content=response_content
        )
        await self.bus.publish(response)

    async def _handle_concept_creation(self, message: Message):
        """Gestionnaire de création de concepts externes"""
        name = message.content.get('name')
        data = message.content.get('data', {})
        plane = message.content.get('plane', 'real')

        if name and name not in self.nodes and name not in self.imagination_nodes:
            await self.add_concept(name, data, plane)

    async def _handle_emergence_event(self, message: Message):
        """Gestionnaire d'événements d'émergence"""
        # Augmente la confiance des concepts liés à l'émergence
        if 'parents' in message.content:
            for parent_name in message.content['parents']:
                node = self.nodes.get(parent_name) or self.imagination_nodes.get(parent_name)
                if node:
                    node.confidence = min(1.0, node.confidence + 0.1)

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du graphe conceptuel"""
        return {
            "real_nodes": len(self.nodes),
            "imagination_nodes": len(self.imagination_nodes),
            "total_connections": sum(len(node.connections) for node in self.nodes.values()),
            "patterns_detected": len(self.detected_patterns),
            "emergence_events": len([p for p in self.pattern_evolution if 'emergence' in str(p)]),
            "average_confidence": np.mean([node.confidence for node in self.nodes.values()]) if self.nodes else 0,
            "imagination_confidence": np.mean([node.confidence for node in self.imagination_nodes.values()]) if self.imagination_nodes else 0
        }

# ============================================================
# CORE COGNITIF AVEC OBJECTIFS ET ADAPTATION
# ============================================================

@dataclass
class CognitiveGoal:
    """Objectif cognitif avec métadonnées"""
    name: str
    description: str
    target_value: float
    current_value: float = 0.0
    priority: int = 1
    strategy: str = "adaptive"
    creation_time: float = field(default_factory=time.time)
    last_update: float = field(default_factory=time.time)
    progress_history: List[float] = field(default_factory=list)

class AdaptiveCognitiveCore:
    """Core cognitif adaptatif avec gestion d'objectifs et méta-cognition"""

    def __init__(self, bus: AsyncMessageBus):
        self.bus = bus
        self.name = "adaptive_core"
        self.start_time = time.time()

        # État cognitif
        self.consciousness_level = 0
        self.attention_focus = []
        self.cognitive_load = 0.0
        self.adaptation_rate = 0.1

        # Objectifs et stratégies
        self.goals: Dict[str, CognitiveGoal] = {}
        self.active_strategies = set()
        self.strategy_effectiveness = defaultdict(float)

        # Modules et behaviours
        self.modules: Dict[str, Any] = {}
        self.behaviours: List[Callable] = []
        self.module_fitness_scores = defaultdict(float)

        # Métriques d'adaptation
        self.adaptation_history = []
        self.performance_metrics = defaultdict(list)
        self.error_count = 0
        self.success_count = 0

        # Ajoute cette ligne à la fin
        self.load_regulator = CognitiveLoadRegulator(self)
        self.regulation_enabled = False

        # Subscriptions
        bus.subscribe("cognitive_request", self._handle_cognitive_request)
        bus.subscribe("adaptation_signal", self._handle_adaptation_signal)
        bus.subscribe("error_event", self._handle_error_event)
        bus.subscribe("success_event", self._handle_success_event)

        # Initialise les objectifs de base
        self._initialize_base_goals()

    def _initialize_base_goals(self):
        """Initialise les objectifs cognitifs de base"""
        self.goals['survive'] = CognitiveGoal(
            name='survive',
            description='Maintenir l\'intégrité et la stabilité du système',
            target_value=1.0,
            priority=1
        )

        self.goals['learn'] = CognitiveGoal(
            name='learn',
            description='Acquérir et intégrer de nouvelles connaissances',
            target_value=0.8,
            priority=2
        )

        self.goals['adapt'] = CognitiveGoal(
            name='adapt',
            description='S\'adapter aux changements environnementaux',
            target_value=0.7,
            priority=2
        )

        self.goals['emerge'] = CognitiveGoal(
            name='emerge',
            description='Développer des capacités émergentes',
            target_value=0.6,
            priority=3
        )

    async def cognitive_cycle(self):
        """Cycle cognitif principal"""
        cycle_start = time.time()

        try:
            # 1. RÉGULATION EN PREMIER (nouveau)
            if self.regulation_enabled and hasattr(self, 'load_regulator'):
                regulation_report = await self.load_regulator.regulate()

                # Log si régulation critique
                if regulation_report.get('severity') == 'critical':
                    logger.warning(f"Critical load regulation: {regulation_report}")

            # 2. Évaluation de l'état actuel
            await self._evaluate_current_state()

            # 3. Mise à jour des objectifs
            await self._update_goals()

            # 4. Sélection et exécution de stratégies
            await self._execute_strategies()

            # 5. Adaptation basée sur les résultats
            await self._adapt_based_on_feedback()

            # 6. Mise à jour du niveau de conscience
            await self._update_consciousness()

            # 7. Maintenance et optimisation
            await self._perform_maintenance()

        except Exception as e:
            logger.error(f"Error in cognitive cycle: {e}")
            self.error_count += 1
            await self._handle_cognitive_error(e)

        # Métriques de performance
        cycle_time = time.time() - cycle_start
        self.performance_metrics['cycle_time'].append(cycle_time)

        # Limite l'historique
        if len(self.performance_metrics['cycle_time']) > 100:
            self.performance_metrics['cycle_time'] = self.performance_metrics['cycle_time'][-50:]

    async def _evaluate_current_state(self):
        """Évalue l'état cognitif actuel"""

        # Calcule la charge cognitive
        total_modules = len(self.modules)
        active_behaviours = len(self.behaviours)
        message_queue_size = len(getattr(self.bus, 'history', []))

        self.cognitive_load = (
            (len(self.active_strategies) * 0.2) +  # Stratégies actives
            (len(self.modules['memory'].hot_cache) / 100 * 0.3) +  # Mémoire
            (message_queue_size * 0.001) +  # Messages en attente
            (len(self.behaviours) * 0.05)  # Comportements
        )

        # Évalue la performance des modules
        for name, module in self.modules.items():
            if hasattr(module, 'get_performance_score'):
                score = await module.get_performance_score()
                self.module_fitness_scores[name] = score

    async def _update_goals(self):
        """Met à jour le progrès des objectifs"""
        current_time = time.time()

        for goal in self.goals.values():
            old_value = goal.current_value

            # Calcule le nouveau progrès basé sur les métriques
            if goal.name == 'survive':
                # Survie basée sur la stabilité du système
                goal.current_value = max(0, 1.0 - (self.error_count / max(1, self.error_count + self.success_count)))

            elif goal.name == 'learn':
                # Apprentissage basé sur l'acquisition de nouvelles connaissances
                if 'memory' in self.modules:
                    memory_stats = self.modules['memory'].get_stats()
                    total_items = memory_stats.get('hot_cache_size', 0) + memory_stats.get('compressed_size', 0)
                    goal.current_value = min(1.0, total_items / 100.0)  # Normalise sur 100 items

            elif goal.name == 'adapt':
                # Adaptation basée sur la variété des stratégies utilisées
                adaptation_score = len(self.active_strategies) / max(1, len(self.strategy_effectiveness))
                goal.current_value = min(1.0, adaptation_score)

            elif goal.name == 'emerge':
                # Émergence basée sur les patterns détectés
                if 'graph' in self.modules:
                    graph_stats = self.modules['graph'].get_stats()
                    emergence_score = graph_stats.get('emergence_events', 0) / 10.0  # Normalise sur 10 événements
                    goal.current_value = min(1.0, emergence_score)

            # Enregistre l'historique
            goal.progress_history.append(goal.current_value)
            goal.last_update = current_time

            # Limite l'historique
            if len(goal.progress_history) > 50:
                goal.progress_history = goal.progress_history[-25:]

            # Détecte les changements significatifs
            if abs(goal.current_value - old_value) > 0.1:
                await self._notify_goal_change(goal, old_value)

    async def _notify_goal_change(self, goal: CognitiveGoal, old_value: float):
        """Notifie un changement significatif d'objectif"""
        message = Message(
            source="cognitive_core",
            target="*",
            msg_type="goal_change",
            content={
                'goal_name': goal.name,
                'old_value': old_value,
                'new_value': goal.current_value,
                'change': goal.current_value - old_value
            },
            semantic_tags=['goal', 'adaptation', 'progress']
        )
        await self.bus.publish(message)

    async def _execute_strategies(self):
        """Sélectionne et exécute les stratégies cognitives"""

        # Sélectionne les objectifs prioritaires non atteints
        priority_goals = sorted(
            [g for g in self.goals.values() if g.current_value < g.target_value],
            key=lambda x: (x.priority, x.target_value - x.current_value),
            reverse=True
        )

        for goal in priority_goals[:3]:  # Limite à 3 objectifs simultanés
            strategy = await self._select_strategy_for_goal(goal)
            if strategy:
                await self._execute_strategy(strategy, goal)

    async def _select_strategy_for_goal(self, goal: CognitiveGoal) -> Optional[str]:
        """Sélectionne la meilleure stratégie pour un objectif"""

        # Stratégies disponibles par objectif
        strategies_map = {
            'survive': ['error_recovery', 'stability_check', 'resource_optimization'],
            'learn': ['knowledge_acquisition', 'pattern_detection', 'memory_consolidation'],
            'adapt': ['strategy_variation', 'parameter_tuning', 'behavior_modification'],
            'emerge': ['connection_exploration', 'imagination_activation', 'pattern_synthesis']
        }

        available_strategies = strategies_map.get(goal.name, [])

        if not available_strategies:
            return None

        # Sélectionne basé sur l'efficacité historique
        best_strategy = None
        best_score = -1

        for strategy in available_strategies:
            effectiveness = self.strategy_effectiveness.get(strategy, 0.5)  # Score par défaut
            randomness = random.random() * 0.2  # Facteur d'exploration
            score = effectiveness + randomness

            if score > best_score:
                best_score = score
                best_strategy = strategy

        return best_strategy

    async def _execute_strategy(self, strategy: str, goal: CognitiveGoal):
        """Exécute une stratégie cognitive spécifique"""

        self.active_strategies.add(strategy)
        execution_start = time.time()
        success = False

        try:
            if strategy == 'error_recovery':
                await self._strategy_error_recovery()
                success = True

            elif strategy == 'knowledge_acquisition':
                await self._strategy_knowledge_acquisition()
                success = True

            elif strategy == 'pattern_detection':
                await self._strategy_pattern_detection()
                success = True

            elif strategy == 'connection_exploration':
                await self._strategy_connection_exploration()
                success = True

            elif strategy == 'imagination_activation':
                await self._strategy_imagination_activation()
                success = True

            # ... autres stratégies peuvent être ajoutées

        except Exception as e:
            logger.error(f"Strategy execution error ({strategy}): {e}")
            success = False

        # Met à jour l'efficacité de la stratégie
        execution_time = time.time() - execution_start
        if success:
            self.strategy_effectiveness[strategy] = min(1.0, self.strategy_effectiveness.get(strategy, 0.5) + 0.1)
            self.success_count += 1
        else:
            self.strategy_effectiveness[strategy] = max(0.0, self.strategy_effectiveness.get(strategy, 0.5) - 0.1)
            self.error_count += 1

        self.active_strategies.discard(strategy)

    async def _strategy_error_recovery(self):
        """Stratégie de récupération d'erreur"""
        # Vérifie l'état des modules
        for name, module in self.modules.items():
            if hasattr(module, 'self_check'):
                check_result = await module.self_check()
                if not check_result:
                    logger.warning(f"Module {name} failed self-check")
                    if hasattr(module, 'restart'):
                        await module.restart()

    async def _strategy_knowledge_acquisition(self):
        """Stratégie d'acquisition de connaissances"""
        # Demande aux modules de partager leurs connaissances récentes
        message = Message(
            source="cognitive_core",
            target="*",
            msg_type="knowledge_request",
            content={'type': 'recent_discoveries'},
            semantic_tags=['knowledge', 'acquisition']
        )
        await self.bus.publish(message)

    async def _strategy_pattern_detection(self):
        """Stratégie de détection de patterns"""
        if 'graph' in self.modules:
            # Demande une détection de patterns au graphe
            message = Message(
                source="cognitive_core",
                target="graph",
                msg_type="graph_query",
                content={'type': 'detect_patterns'},
                semantic_tags=['pattern', 'detection']
            )
            await self.bus.publish(message)

    async def _strategy_connection_exploration(self):
        """Stratégie d'exploration de connexions"""
        if 'graph' in self.modules:
            # Explore de nouvelles connexions potentielles
            graph = self.modules['graph']
            if hasattr(graph, 'nodes') and len(graph.nodes) > 1:
                # Sélectionne deux concepts aléatoirement pour explorer leur connexion
                concepts = list(graph.nodes.keys())
                if len(concepts) >= 2:
                    concept1, concept2 = random.sample(concepts, 2)
                    await graph.connect_concepts(concept1, concept2, 0.3)

    async def _strategy_imagination_activation(self):
        """Stratégie d'activation de l'imagination"""
        if 'graph' in self.modules:
            graph = self.modules['graph']
            if hasattr(graph, 'add_concept'):
                # Crée un concept imaginaire
                imaginary_concept = f"imagine_{int(time.time())}"
                imaginary_data = {
                    'type': 'imaginative',
                    'source': 'cognitive_strategy',
                    'confidence': 0.3
                }
                await graph.add_concept(imaginary_concept, imaginary_data, plane="imagination")

    async def _adapt_based_on_feedback(self):
        """Adaptation basée sur le feedback des actions"""

        # Calcule le taux de succès récent
        total_actions = self.success_count + self.error_count
        if total_actions > 0:
            success_rate = self.success_count / total_actions

            # Adapte le taux d'adaptation basé sur le succès
            if success_rate > 0.7:
                self.adaptation_rate = max(0.05, self.adaptation_rate - 0.01)  # Ralentit si bon
            elif success_rate < 0.3:
                self.adaptation_rate = min(0.3, self.adaptation_rate + 0.02)   # Accélère si mauvais

        # Enregistre l'adaptation
        self.adaptation_history.append({
            'timestamp': time.time(),
            'adaptation_rate': self.adaptation_rate,
            'success_rate': self.success_count / max(1, total_actions),
            'cognitive_load': self.cognitive_load
        })

        # Limite l'historique
        if len(self.adaptation_history) > 100:
            self.adaptation_history = self.adaptation_history[-50:]

    async def _update_consciousness(self):
        """Met à jour le niveau de conscience basé sur l'activité cognitive"""

        # Facteurs influençant la conscience
        goal_achievement = np.mean([g.current_value for g in self.goals.values()])
        module_activity = len(self.active_strategies) / 5.0  # Normalise sur 5 stratégies max
        adaptation_activity = min(1.0, self.adaptation_rate * 10)  # Normalise

        # Calcule le nouveau niveau de conscience
        new_consciousness = (
            goal_achievement * 0.4 +
            module_activity * 0.3 +
            adaptation_activity * 0.3
        )

        # Filtre pour éviter les changements trop brusques
        self.consciousness_level = (
            self.consciousness_level * 0.8 +
            new_consciousness * 0.2
        )

        # Limite entre 0 et 1
        self.consciousness_level = max(0, min(1, self.consciousness_level))

    async def _perform_maintenance(self):
        """Effectue la maintenance cognitive"""

        # Nettoie les anciennes métriques
        current_time = time.time()

        # Nettoie l'historique d'adaptation ancien (plus de 1 heure)
        self.adaptation_history = [
            h for h in self.adaptation_history
            if current_time - h['timestamp'] < 3600
        ]

        # Reset périodique des compteurs d'erreur/succès
        if current_time % 300 < 1:  # Tous les 5 minutes
            self.error_count = int(self.error_count * 0.9)
            self.success_count = int(self.success_count * 0.9)

        # Sauvegarde périodique de l'état
        if current_time % 600 < 1:  # Tous les 10 minutes
            await self._save_cognitive_state()

    async def _save_cognitive_state(self):
        """Sauvegarde l'état cognitif"""
        state = {
            'consciousness_level': self.consciousness_level,
            'goals': {name: {
                'current_value': goal.current_value,
                'progress_history': goal.progress_history[-10:]  # Derniers 10 points
            } for name, goal in self.goals.items()},
            'strategy_effectiveness': dict(self.strategy_effectiveness),
            'adaptation_rate': self.adaptation_rate,
            'module_fitness': dict(self.module_fitness_scores)
        }

        try:
            with open('cognitive_state.json', 'w') as f:
                json.dump(state, f, indent=2, default=str)
            logger.info("Cognitive state saved")
        except Exception as e:
            logger.error(f"Error saving cognitive state: {e}")

    async def _handle_cognitive_error(self, error: Exception):
        """Gère les erreurs cognitives"""
        # Stratégie de récupération d'urgence
        self.cognitive_load = min(1.0, self.cognitive_load + 0.2)

        # Notifie l'erreur
        message = Message(
            source="cognitive_core",
            target="*",
            msg_type="error_event",
            content={
                'error_type': type(error).__name__,
                'error_message': str(error),
                'cognitive_load': self.cognitive_load
            },
            semantic_tags=['error', 'recovery']
        )
        await self.bus.publish(message)

    async def _handle_cognitive_request(self, message: Message):
        """Gestionnaire des requêtes cognitives"""
        request_type = message.content.get('type')

        if request_type == 'status':
            response_content = {
                'consciousness_level': self.consciousness_level,
                'cognitive_load': self.cognitive_load,
                'active_strategies': list(self.active_strategies),
                'goals_progress': {name: goal.current_value for name, goal in self.goals.items()}
            }
        elif request_type == 'adapt':
            await self._adapt_based_on_feedback()
            response_content = {'adaptation_triggered': True}
        else:
            response_content = {'error': 'Unknown request type'}

        response = Message(
            source="cognitive_core",
            target=message.source,
            msg_type="cognitive_response",
            content=response_content
        )
        await self.bus.publish(response)

    async def _handle_adaptation_signal(self, message: Message):
        """Gestionnaire des signaux d'adaptation"""
        signal_type = message.content.get('signal_type')

        if signal_type == 'performance_change':
            # Adapte basé sur un changement de performance
            performance_delta = message.content.get('delta', 0)
            if performance_delta < 0:
                self.adaptation_rate = min(0.3, self.adaptation_rate + 0.05)
            else:
                self.adaptation_rate = max(0.05, self.adaptation_rate - 0.02)

    async def _handle_error_event(self, message: Message):
        """Gestionnaire d'événements d'erreur"""
        self.error_count += 1
        # Déclenche une stratégie de récupération
        await self._execute_strategy('error_recovery', self.goals['survive'])

    async def _handle_success_event(self, message: Message):
        """Gestionnaire d'événements de succès"""
        self.success_count += 1

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du core cognitif"""
        return {
            "consciousness_level": self.consciousness_level,
            "cognitive_load": self.cognitive_load,
            "adaptation_rate": self.adaptation_rate,
            "goals_progress": {name: goal.current_value for name, goal in self.goals.items()},
            "active_strategies": list(self.active_strategies),
            "strategy_count": len(self.strategy_effectiveness),
            "success_rate": self.success_count / max(1, self.success_count + self.error_count),
            "module_count": len(self.modules),
            "behaviour_count": len(self.behaviours),
            "uptime": time.time() - self.start_time
        }

# ============================================================
# SYSTÈME DE RÉGULATION DE CHARGE COGNITIVE
# À intégrer dans AdaptiveCognitiveCore
# ============================================================

import asyncio
import logging
from typing import Dict, Any, List
from collections import deque
import numpy as np

logger = logging.getLogger("GrandNexus.CognitiveRegulator")

class CognitiveLoadRegulator:
    """
    Système de régulation proportionnelle de la charge cognitive
    Maintient le système dans une zone de performance optimale
    """

    def __init__(self, cognitive_core):
        self.core = cognitive_core

        # Paramètres de régulation
        self.target_load = 0.7  # Charge optimale
        self.max_safe_load = 1.2  # Seuil critique
        self.min_load = 0.3  # Seuil de sous-utilisation

        # Vitesses d'adaptation
        self.aggressive_reduction_rate = 0.4  # Réduction rapide en urgence
        self.gentle_reduction_rate = 0.1  # Réduction progressive
        self.expansion_rate = 0.05  # Augmentation prudente

        # Historique pour analyse de tendance
        self.load_history = deque(maxlen=20)
        self.regulation_history = []

        # Compteurs de régulation
        self.total_regulations = 0
        self.emergency_regulations = 0
        self.gentle_regulations = 0

        # État
        self.last_regulation_time = 0
        self.regulation_cooldown = 2.0  # secondes entre régulations

    async def regulate(self) -> Dict[str, Any]:
        """
        Effectue un cycle de régulation
        Returns: Rapport de régulation
        """
        import time
        current_time = time.time()

        # Vérifie le cooldown
        if current_time - self.last_regulation_time < self.regulation_cooldown:
            return {'action': 'cooldown', 'reason': 'too_soon'}

        current_load = self.core.cognitive_load
        self.load_history.append(current_load)

        regulation_report = {
            'timestamp': current_time,
            'load_before': current_load,
            'load_target': self.target_load,
            'action': 'none',
            'interventions': []
        }

        # Analyse de la tendance
        load_trend = self._calculate_load_trend()

        # Décision de régulation basée sur le niveau et la tendance
        if current_load > self.max_safe_load:
            # URGENCE : Réduction agressive
            await self._emergency_reduction(regulation_report)
            self.emergency_regulations += 1

        elif current_load > self.target_load:
            # Charge élevée : Réduction douce
            if load_trend > 0:  # Et la charge augmente encore
                await self._gentle_reduction(regulation_report)
                self.gentle_regulations += 1
            else:
                regulation_report['action'] = 'monitor'
                regulation_report['reason'] = 'load_high_but_decreasing'

        elif current_load < self.min_load:
            # Sous-utilisation : Augmentation de complexité
            await self._increase_complexity(regulation_report)

        else:
            # Zone optimale
            regulation_report['action'] = 'optimal'
            regulation_report['reason'] = 'load_in_target_range'

        # Enregistrement
        regulation_report['load_after'] = self.core.cognitive_load
        regulation_report['load_change'] = regulation_report['load_after'] - regulation_report['load_before']
        regulation_report['trend'] = load_trend

        self.regulation_history.append(regulation_report)
        self.total_regulations += 1
        self.last_regulation_time = current_time

        # Log si action prise
        if regulation_report['action'] not in ['none', 'optimal', 'monitor', 'cooldown']:
            logger.info(f"🔧 Regulation: {regulation_report['action']} | "
                       f"Load: {regulation_report['load_before']:.2f} → {regulation_report['load_after']:.2f}")

        return regulation_report

    def _calculate_load_trend(self) -> float:
        """Calcule la tendance de la charge (montée/descente)"""
        if len(self.load_history) < 3:
            return 0.0

        recent = list(self.load_history)[-5:]

        # Régression linéaire simple
        n = len(recent)
        x = list(range(n))
        mean_x = sum(x) / n
        mean_y = sum(recent) / n

        numerator = sum((x[i] - mean_x) * (recent[i] - mean_y) for i in range(n))
        denominator = sum((x[i] - mean_x) ** 2 for i in range(n))

        if denominator == 0:
            return 0.0

        slope = numerator / denominator
        return slope

    async def _emergency_reduction(self, report: Dict[str, Any]):
        """Réduction agressive en situation d'urgence"""
        report['action'] = 'emergency_reduction'
        report['severity'] = 'critical'
        interventions = []

        # 1. Réduction drastique des stratégies actives
        if hasattr(self.core, 'active_strategies'):
            original_count = len(self.core.active_strategies)
            if original_count > 1:
                # Ne garde que la stratégie la plus efficace
                best_strategy = max(
                    self.core.active_strategies,
                    key=lambda s: self.core.strategy_effectiveness.get(s, 0)
                ) if self.core.active_strategies else None

                self.core.active_strategies = {best_strategy} if best_strategy else set()
                interventions.append({
                    'type': 'strategy_reduction',
                    'from': original_count,
                    'to': len(self.core.active_strategies)
                })

        # 2. Compression agressive de la mémoire
        if 'memory' in self.core.modules:
            memory = self.core.modules['memory']
            if hasattr(memory, 'hot_cache'):
                original_size = len(memory.hot_cache)

                # Force l'éviction des items les moins importants
                items_to_keep = int(original_size * (1 - self.aggressive_reduction_rate))

                # Trie par importance et garde les meilleurs
                sorted_items = sorted(
                    memory.hot_cache.items(),
                    key=lambda x: getattr(x[1], 'importance', 0) * getattr(x[1], 'access_count', 0),
                    reverse=True
                )

                # Reconstruit le cache avec seulement les items importants
                from collections import OrderedDict
                memory.hot_cache = OrderedDict(sorted_items[:items_to_keep])

                interventions.append({
                    'type': 'memory_compression',
                    'from': original_size,
                    'to': len(memory.hot_cache),
                    'freed': original_size - len(memory.hot_cache)
                })

        # 3. Élagage du graphe conceptuel
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']
            if hasattr(graph, 'imagination_nodes'):
                original_count = len(graph.imagination_nodes)

                # Supprime tous les concepts imagination avec faible confiance
                to_remove = [
                    name for name, node in graph.imagination_nodes.items()
                    if getattr(node, 'confidence', 0) < 0.3
                ]

                for name in to_remove:
                    del graph.imagination_nodes[name]

                interventions.append({
                    'type': 'imagination_pruning',
                    'removed': len(to_remove),
                    'remaining': len(graph.imagination_nodes)
                })

        # 4. Reset des compteurs d'erreur pour éviter la spirale
        if hasattr(self.core, 'error_count'):
            old_errors = self.core.error_count
            self.core.error_count = int(self.core.error_count * 0.5)
            interventions.append({
                'type': 'error_reset',
                'from': old_errors,
                'to': self.core.error_count
            })

        # 5. Réduction du taux d'adaptation pour stabiliser
        if hasattr(self.core, 'adaptation_rate'):
            old_rate = self.core.adaptation_rate
            self.core.adaptation_rate = max(0.05, self.core.adaptation_rate * 0.7)
            interventions.append({
                'type': 'adaptation_slowdown',
                'from': old_rate,
                'to': self.core.adaptation_rate
            })

        report['interventions'] = interventions
        logger.warning(f"⚠️ EMERGENCY REDUCTION: {len(interventions)} interventions applied")

    async def _gentle_reduction(self, report: Dict[str, Any]):
        """Réduction douce et progressive"""
        report['action'] = 'gentle_reduction'
        report['severity'] = 'moderate'
        interventions = []

        # 1. Réduction progressive des stratégies
        if hasattr(self.core, 'active_strategies') and len(self.core.active_strategies) > 2:
            # Enlève la stratégie la moins efficace
            worst_strategy = min(
                self.core.active_strategies,
                key=lambda s: self.core.strategy_effectiveness.get(s, 0.5)
            )
            self.core.active_strategies.discard(worst_strategy)
            interventions.append({
                'type': 'strategy_removal',
                'removed': worst_strategy,
                'remaining': len(self.core.active_strategies)
            })

        # 2. Consolidation de mémoire douce
        if 'memory' in self.core.modules:
            memory = self.core.modules['memory']
            if hasattr(memory, 'hot_cache') and len(memory.hot_cache) > 20:
                # Éviction des 20% items les moins utilisés
                items_to_evict = int(len(memory.hot_cache) * 0.2)

                sorted_items = sorted(
                    memory.hot_cache.items(),
                    key=lambda x: getattr(x[1], 'access_count', 0)
                )

                # Éviction
                for key, _ in sorted_items[:items_to_evict]:
                    memory.hot_cache.pop(key, None)

                interventions.append({
                    'type': 'memory_eviction',
                    'evicted': items_to_evict,
                    'remaining': len(memory.hot_cache)
                })

        # 3. Decay de l'imagination
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']
            if hasattr(graph, 'decay_imagination'):
                await graph.decay_imagination()
                interventions.append({
                    'type': 'imagination_decay',
                    'remaining': len(getattr(graph, 'imagination_nodes', {}))
                })

        report['interventions'] = interventions

    async def _increase_complexity(self, report: Dict[str, Any]):
        """Augmente la complexité quand le système est sous-utilisé"""
        report['action'] = 'complexity_increase'
        report['severity'] = 'low'
        interventions = []

        # 1. Activation d'une nouvelle stratégie
        if hasattr(self.core, 'strategy_effectiveness'):
            available_strategies = [
                s for s in self.core.strategy_effectiveness.keys()
                if s not in getattr(self.core, 'active_strategies', set())
            ]

            if available_strategies and len(self.core.active_strategies) < 4:
                # Choisit une stratégie avec bonne efficacité mais pas utilisée
                new_strategy = max(
                    available_strategies,
                    key=lambda s: self.core.strategy_effectiveness.get(s, 0.5)
                )
                self.core.active_strategies.add(new_strategy)
                interventions.append({
                    'type': 'strategy_activation',
                    'added': new_strategy,
                    'total': len(self.core.active_strategies)
                })

        # 2. Stimulation de l'imagination
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']

            # Crée quelques concepts imaginaires pour stimuler l'exploration
            import time
            for i in range(2):
                concept_name = f"explore_{int(time.time())}_{i}"
                concept_data = {
                    'type': 'exploratory',
                    'confidence': 0.4,
                    'source': 'complexity_increase'
                }
                if hasattr(graph, 'add_concept'):
                    await graph.add_concept(concept_name, concept_data, plane='imagination')

            interventions.append({
                'type': 'imagination_stimulation',
                'added_concepts': 2
            })

        # 3. Augmentation douce du taux d'adaptation
        if hasattr(self.core, 'adaptation_rate'):
            old_rate = self.core.adaptation_rate
            self.core.adaptation_rate = min(0.3, self.core.adaptation_rate * 1.1)
            interventions.append({
                'type': 'adaptation_acceleration',
                'from': old_rate,
                'to': self.core.adaptation_rate
            })

        report['interventions'] = interventions

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du régulateur"""
        return {
            'total_regulations': self.total_regulations,
            'emergency_regulations': self.emergency_regulations,
            'gentle_regulations': self.gentle_regulations,
            'current_load': self.core.cognitive_load,
            'target_load': self.target_load,
            'load_trend': self._calculate_load_trend(),
            'avg_load': np.mean(list(self.load_history)) if self.load_history else 0,
            'load_stability': np.std(list(self.load_history)) if len(self.load_history) > 1 else 0
        }


# ============================================================
# ENVIRONNEMENT ADAPTATIF MULTI-MODAL
# ============================================================

class AdaptiveEnvironment:
    """Environnement adaptatif avec stimuli multi-modaux"""

    def __init__(self, bus: AsyncMessageBus):
        self.bus = bus
        self.name = "adaptive_environment"
        self.running = False

        # Sources de stimuli multiples
        self.stimulus_sources = {
            'numerical': self._generate_numerical_stimulus,
            'textual': self._generate_textual_stimulus,
            'temporal': self._generate_temporal_stimulus,
            'relational': self._generate_relational_stimulus
        }

        # Configuration adaptative
        self.stimulus_intensity = 0.5
        self.stimulus_variety = 0.7
        self.adaptation_sensitivity = 0.1

        # Datasets intégrés
        self.knowledge_fragments = [
            "pattern recognition enables learning",
            "memory compression preserves essence",
            "emergence arises from interaction",
            "consciousness requires recursive observation",
            "adaptation follows environmental pressure",
            "complexity emerges from simple rules",
            "feedback loops amplify small changes",
            "self-organization creates order from chaos",
            "information theory guides understanding",
            "cognitive architectures model thinking"
        ]

        self.numerical_sequences = [
            [1, 1, 2, 3, 5, 8, 13],  # Fibonacci
            [1, 4, 9, 16, 25, 36],   # Squares
            [2, 4, 8, 16, 32, 64],   # Powers of 2
            [1, 3, 6, 10, 15, 21],   # Triangular numbers
        ]

        # État environnemental
        self.environmental_state = {
            'temperature': 20.0,
            'complexity': 0.3,
            'novelty': 0.5,
            'coherence': 0.8
        }

        # Historique et adaptation
        self.stimulus_history = []
        self.response_feedback = []

        # Subscriptions
        bus.subscribe("environment_adaptation", self._handle_adaptation_request)
        bus.subscribe("stimulus_feedback", self._handle_stimulus_feedback)

    async def start(self):
        """Démarre l'environnement adaptatif"""
        self.running = True
        logger.info("Adaptive environment started")

    async def stop(self):
        """Arrête l'environnement"""
        self.running = False
        logger.info("Adaptive environment stopped")

    async def perceive(self) -> Optional[Dict[str, Any]]:
        """Génère une perception adaptative multi-modale"""

        if not self.running:
            return None

        # Sélectionne les modalités de stimulus basé sur la variété
        active_modalities = self._select_active_modalities()

        perception = {
            'timestamp': time.time(),
            'environmental_state': dict(self.environmental_state),
            'modalities': {}
        }

        # Génère les stimuli pour chaque modalité active
        for modality in active_modalities:
            if modality in self.stimulus_sources:
                stimulus = await self.stimulus_sources[modality]()
                perception['modalities'][modality] = stimulus

        # Ajoute des métadonnées adaptatives
        perception['meta'] = {
            'intensity': self.stimulus_intensity,
            'variety': len(active_modalities),
            'complexity_hint': self.environmental_state['complexity']
        }

        # Enregistre dans l'historique
        self.stimulus_history.append(perception)
        if len(self.stimulus_history) > 100:
            self.stimulus_history = self.stimulus_history[-50:]

        # Met à jour l'état environnemental
        await self._update_environmental_state()

        return perception

    def _select_active_modalities(self) -> List[str]:
        """Sélectionne les modalités actives basé sur la variété configurée"""
        all_modalities = list(self.stimulus_sources.keys())
        num_active = max(1, int(len(all_modalities) * self.stimulus_variety))

        # Sélection pondérée par l'intensité
        if self.stimulus_intensity > 0.7:
            return random.sample(all_modalities, min(num_active + 1, len(all_modalities)))
        else:
            return random.sample(all_modalities, num_active)

    async def _generate_numerical_stimulus(self) -> Dict[str, Any]:
        """Génère un stimulus numérique"""

        # Sélectionne un type de stimulus numérique
        stimulus_type = random.choice(['sequence', 'pattern', 'random', 'function'])

        if stimulus_type == 'sequence':
            sequence = random.choice(self.numerical_sequences)
            # Ajoute quelques éléments ou altère légèrement
            if random.random() > 0.5:
                next_val = sequence[-1] + sequence[-2] if len(sequence) > 1 else sequence[-1] * 2
                sequence = sequence + [next_val]

            return {
                'type': 'sequence',
                'data': sequence,
                'challenge': 'continue_sequence'
            }

        elif stimulus_type == 'pattern':
            base = random.randint(2, 5)
            pattern = [base ** i for i in range(4)]
            return {
                'type': 'pattern',
                'data': pattern,
                'base': base,
                'challenge': 'identify_pattern'
            }

        elif stimulus_type == 'function':
            x = random.randint(1, 10)
            operations = [
                ('square', x**2),
                ('double', x*2),
                ('increment', x+1),
                ('fibonacci_pos', self._fibonacci(x))
            ]
            op_name, result = random.choice(operations)

            return {
                'type': 'function',
                'input': x,
                'output': result,
                'operation': op_name,
                'challenge': 'deduce_function'
            }

        else:  # random
            return {
                'type': 'random',
                'data': [random.randint(0, 100) for _ in range(random.randint(3, 8))],
                'challenge': 'find_meaning'
            }

    async def _generate_textual_stimulus(self) -> Dict[str, Any]:
        """Génère un stimulus textuel"""

        fragment = random.choice(self.knowledge_fragments)

        # Variations du stimulus textuel
        variations = ['complete', 'partial', 'keywords', 'conceptual']
        variation = random.choice(variations)

        if variation == 'partial':
            words = fragment.split()
            # Masque quelques mots
            masked_indices = random.sample(range(len(words)), min(2, len(words)//3))
            for idx in masked_indices:
                words[idx] = "[MASK]"
            processed_text = " ".join(words)

        elif variation == 'keywords':
            words = fragment.split()
            # Garde seulement les mots significatifs
            significant_words = [w for w in words if len(w) > 3 and w.lower() not in
                               ['from', 'with', 'that', 'this', 'they', 'have', 'been']]
            processed_text = " ".join(significant_words[:4])

        elif variation == 'conceptual':
            # Extrait les concepts principaux
            concepts = self._extract_concepts_from_text(fragment)
            processed_text = " + ".join(concepts[:3])

        else:  # complete
            processed_text = fragment

        return {
            'type': 'textual',
            'original': fragment,
            'processed': processed_text,
            'variation': variation,
            'challenge': 'interpret_meaning'
        }

    async def _generate_temporal_stimulus(self) -> Dict[str, Any]:
        """Génère un stimulus temporel"""

        current_time = time.time()

        # Types de stimuli temporels
        temporal_types = ['rhythm', 'sequence', 'cycle', 'trend']
        temporal_type = random.choice(temporal_types)

        if temporal_type == 'rhythm':
            # Génère un rythme simple
            beats = random.choice([
                [1, 0, 1, 0, 1, 0, 1, 0],  # Régulier
                [1, 1, 0, 1, 0, 1, 1, 0],  # Irrégulier
                [1, 0, 0, 1, 0, 0, 1, 0]   # Ternaire
            ])
            return {
                'type': 'rhythm',
                'pattern': beats,
                'timestamp': current_time,
                'challenge': 'predict_next_beat'
            }

        elif temporal_type == 'sequence':
            # Séquence temporelle avec intervalles
            intervals = [1, 2, 4, 8]  # Doublement
            events = [current_time + sum(intervals[:i+1]) for i in range(len(intervals))]
            return {
                'type': 'sequence',
                'events': events,
                'intervals': intervals,
                'challenge': 'predict_next_event'
            }

        elif temporal_type == 'cycle':
            # Cycle temporel
            cycle_length = random.choice([5, 7, 10])
            phase = int(current_time) % cycle_length
            return {
                'type': 'cycle',
                'cycle_length': cycle_length,
                'current_phase': phase,
                'progress': phase / cycle_length,
                'challenge': 'identify_cycle'
            }

        else:  # trend
            # Tendance temporelle
            trend_data = [20 + i*2 + random.uniform(-1, 1) for i in range(10)]
            return {
                'type': 'trend',
                'data': trend_data,
                'slope': 2,  # Tendance croissante
                'noise_level': 0.5,
                'challenge': 'extrapolate_trend'
            }

    async def _generate_relational_stimulus(self) -> Dict[str, Any]:
        """Génère un stimulus relationnel"""

        # Types de relations
        relation_types = ['hierarchy', 'causality', 'similarity', 'opposition']
        relation_type = random.choice(relation_types)

        if relation_type == 'hierarchy':
            hierarchy = {
                'root': 'system',
                'children': {
                    'cognition': ['perception', 'reasoning', 'memory'],
                    'behavior': ['action', 'response', 'adaptation'],
                    'environment': ['input', 'feedback', 'context']
                }
            }
            return {
                'type': 'hierarchy',
                'structure': hierarchy,
                'challenge': 'navigate_hierarchy'
            }

        elif relation_type == 'causality':
            causal_chain = [
                ('stimulus', 'perception'),
                ('perception', 'processing'),
                ('processing', 'decision'),
                ('decision', 'action'),
                ('action', 'feedback')
            ]
            return {
                'type': 'causality',
                'chain': causal_chain,
                'challenge': 'trace_causality'
            }

        elif relation_type == 'similarity':
            similar_groups = [
                ['learning', 'adaptation', 'evolution'],
                ['pattern', 'structure', 'organization'],
                ['memory', 'storage', 'persistence']
            ]
            group = random.choice(similar_groups)
            return {
                'type': 'similarity',
                'group': group,
                'common_trait': 'information_processing',
                'challenge': 'find_similarity'
            }

        else:  # opposition
            oppositions = [
                ('order', 'chaos'),
                ('simple', 'complex'),
                ('static', 'dynamic'),
                ('local', 'global')
            ]
            opposition = random.choice(oppositions)
            return {
                'type': 'opposition',
                'pair': opposition,
                'dimension': 'conceptual_axis',
                'challenge': 'balance_opposites'
            }

    def _extract_concepts_from_text(self, text: str) -> List[str]:
        """Extrait les concepts clés d'un texte"""
        words = text.lower().split()
        # Filtre et garde les mots significatifs
        concepts = [w for w in words if len(w) > 3 and w.isalpha()]
        return concepts[:5]  # Top 5 concepts

    def _fibonacci(self, n: int) -> int:
        """Calcule le n-ième nombre de Fibonacci"""
        if n <= 1:
            return n
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b

    async def _update_environmental_state(self):
        """Met à jour l'état environnemental basé sur l'activité"""

        # Calcule la complexité basée sur l'historique récent
        if len(self.stimulus_history) > 5:
            recent_stimuli = self.stimulus_history[-5:]
            modality_variety = len(set(
                modality for stimulus in recent_stimuli
                for modality in stimulus.get('modalities', {}).keys()
            ))
            self.environmental_state['complexity'] = min(1.0, modality_variety / 4.0)

        # Met à jour la nouveauté
        self.environmental_state['novelty'] = random.uniform(0.3, 0.8)

        # Ajuste la température en fonction de l'activité
        activity_level = len(self.stimulus_history) / 100.0
        self.environmental_state['temperature'] = 20.0 + activity_level * 10

        # Cohérence basée sur les feedbacks
        if self.response_feedback:
            positive_feedback = sum(1 for f in self.response_feedback[-10:] if f.get('positive', False))
            self.environmental_state['coherence'] = positive_feedback / min(10, len(self.response_feedback))

    async def _handle_adaptation_request(self, message: Message):
        """Gestionnaire des requêtes d'adaptation environnementale"""
        adaptation_type = message.content.get('type')

        if adaptation_type == 'increase_complexity':
            self.stimulus_intensity = min(1.0, self.stimulus_intensity + 0.1)
            self.stimulus_variety = min(1.0, self.stimulus_variety + 0.1)

        elif adaptation_type == 'decrease_complexity':
            self.stimulus_intensity = max(0.1, self.stimulus_intensity - 0.1)
            self.stimulus_variety = max(0.2, self.stimulus_variety - 0.1)

        elif adaptation_type == 'reset':
            self.stimulus_intensity = 0.5
            self.stimulus_variety = 0.7

        logger.info(f"Environment adapted: intensity={self.stimulus_intensity:.2f}, variety={self.stimulus_variety:.2f}")

    async def _handle_stimulus_feedback(self, message: Message):
        """Gestionnaire du feedback sur les stimuli"""
        feedback = message.content
        self.response_feedback.append(feedback)

        # Limite l'historique du feedback
        if len(self.response_feedback) > 50:
            self.response_feedback = self.response_feedback[-25:]

        # Adaptation basée sur le feedback
        if feedback.get('too_complex', False):
            await self._handle_adaptation_request(Message(
                source="feedback",
                target="environment",
                msg_type="environment_adaptation",
                content={'type': 'decrease_complexity'}
            ))
        elif feedback.get('too_simple', False):
            await self._handle_adaptation_request(Message(
                source="feedback",
                target="environment",
                msg_type="environment_adaptation",
                content={'type': 'increase_complexity'}
            ))

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques de l'environnement"""
        return {
            "stimulus_intensity": self.stimulus_intensity,
            "stimulus_variety": self.stimulus_variety,
            "environmental_state": dict(self.environmental_state),
            "stimulus_count": len(self.stimulus_history),
            "feedback_count": len(self.response_feedback),
            "modalities_available": len(self.stimulus_sources),
            "knowledge_fragments": len(self.knowledge_fragments),
            "running": self.running
        }

# ============================================================
# ORCHESTRATEUR PRINCIPAL - VERSION COLAB OPTIMISÉE
# ============================================================

class GrandNexusMiniOrchestrator:
    """Orchestrateur principal pour GrandNexus Mini optimisé pour Colab"""

    def __init__(self):
        self.bus = None
        self.environment = None
        self.memory = None
        self.graph = None
        self.cognitive_core = None
        self.running = False
        self.start_time = None

        # Métriques globales
        self.cycle_count = 0
        self.total_messages = 0
        self.performance_history = []

    async def initialize(self):
        """Initialise tous les composants"""
        logger.info("🚀 Initializing GrandNexus Mini Enhanced...")

        # 1. Message Bus
        self.bus = AsyncMessageBus()
        await self.bus.start()
        logger.info("✅ Message bus initialized")

        # 2. Environment
        self.environment = AdaptiveEnvironment(self.bus)
        await self.environment.start()
        logger.info("✅ Adaptive environment initialized")

        # 3. Memory
        self.memory = EnhancedHierarchicalMemory(self.bus)
        logger.info("✅ Enhanced memory initialized")

        # 4. Conceptual Graph
        self.graph = EmergentConceptualGraph(self.bus)
        logger.info("✅ Emergent graph initialized")

        # 5. Cognitive Core
        self.cognitive_core = AdaptiveCognitiveCore(self.bus)

        # Register modules in cognitive core
        self.cognitive_core.modules = {
            'memory': self.memory,
            'graph': self.graph,
            'environment': self.environment
        }
        logger.info("✅ Cognitive core initialized")

        # 6. Initial concepts
        await self._initialize_base_concepts()
        logger.info("✅ Base concepts initialized")


        # 7. 🆕 Consolidation des connaissances
        self.consolidator = KnowledgeConsolidator(self)
        logger.info("✅ Knowledge consolidator initialized")

        self.start_time = time.time()
        logger.info("🎯 GrandNexus Mini Enhanced ready!")



    async def _initialize_base_concepts(self):
        """Initialise les concepts de base"""
        base_concepts = [
            ('self', {'type': 'identity', 'importance': 1.0}),
            ('environment', {'type': 'external', 'importance': 0.8}),
            ('learning', {'type': 'process', 'importance': 0.9}),
            ('adaptation', {'type': 'capability', 'importance': 0.8}),
            ('emergence', {'type': 'phenomenon', 'importance': 0.7})
        ]

        for name, data in base_concepts:
            await self.graph.add_concept(name, data)
            await self.memory.store(
                f"concept_{name}",
                data,
                importance=data.get('importance', 0.5),
                semantic_tags=['concept', 'base', data.get('type', 'unknown')]
            )

    async def run_cycle(self):
        """Exécute un cycle complet du système"""
        cycle_start = time.time()

        try:
            # 🆕 CONSOLIDATION CHECK (ajoute au début)
            if hasattr(self, 'consolidator'):
                if await self.consolidator.should_consolidate():
                    consolidation_report = await self.consolidator.consolidate()
                    # Log important
                    logger.info(f"🔄 Consolidation: {consolidation_report['improvements']}")




            # 1. Perception environnementale
            perception = await self.environment.perceive()
            if perception:
                await self._process_perception(perception)

            # 2. Cycle cognitif
            await self.cognitive_core.cognitive_cycle()

            # 3. Processus de mémoire
            await self._memory_consolidation()

            # 4. Évolution du graphe
            await self._graph_evolution()

            # 5. Métriques et adaptation
            await self._update_metrics()

            self.cycle_count += 1

        except Exception as e:
            logger.error(f"Error in system cycle: {e}")
            # Stratégie de récupération simple
            await asyncio.sleep(0.1)

        cycle_time = time.time() - cycle_start
        self.performance_history.append(cycle_time)

        # Limite l'historique
        if len(self.performance_history) > 100:
            self.performance_history = self.performance_history[-50:]

    async def _process_perception(self, perception: Dict[str, Any]):
        """Traite une perception"""

        # Stocke la perception en mémoire
        perception_key = f"perception_{int(perception['timestamp'])}"
        await self.memory.store(
            perception_key,
            perception,
            importance=0.6,
            semantic_tags=['perception', 'input'],
            provenance='environment'
        )

        # Traite chaque modalité
        for modality, stimulus in perception.get('modalities', {}).items():
            await self._process_stimulus(modality, stimulus)

    async def _process_stimulus(self, modality: str, stimulus: Dict[str, Any]):
        """Traite un stimulus spécifique"""

        if modality == 'textual':
            # Extrait et stocke les concepts textuels
            concepts = self._extract_concepts_from_text(stimulus.get('processed', ''))
            for concept in concepts:
                await self.graph.add_concept(
                    f"text_concept_{concept}",
                    {'type': 'textual_concept', 'value': concept},
                    plane='real'
                )

        elif modality == 'numerical':
            # Analyse les patterns numériques
            if stimulus.get('type') == 'sequence':
                pattern_key = f"num_pattern_{int(time.time())}"
                await self.memory.store(
                    pattern_key,
                    stimulus,
                    importance=0.7,
                    semantic_tags=['pattern', 'numerical']
                )

        elif modality == 'relational':
            # Traite les relations
            if stimulus.get('type') == 'hierarchy':
                await self._process_hierarchy(stimulus.get('structure', {}))

    async def _process_hierarchy(self, hierarchy: Dict[str, Any]):
        """Traite une structure hiérarchique"""
        root = hierarchy.get('root')
        if root:
            await self.graph.add_concept(root, {'type': 'hierarchy_root'})

            children = hierarchy.get('children', {})
            for parent, child_list in children.items():
                await self.graph.add_concept(parent, {'type': 'hierarchy_node'})
                await self.graph.connect_concepts(root, parent, 0.8)

                for child in child_list:
                    await self.graph.add_concept(child, {'type': 'hierarchy_leaf'})
                    await self.graph.connect_concepts(parent, child, 0.6)

    def _extract_concepts_from_text(self, text: str) -> List[str]:
        """Extrait les concepts d'un texte"""
        if not text:
            return []
        words = text.lower().split()
        return [w for w in words if len(w) > 3 and w.isalpha()][:3]

    async def _memory_consolidation(self):
        """Consolide la mémoire périodiquement"""
        if self.cycle_count % 20 == 0:  # Tous les 20 cycles
            # Recherche sémantique pour renforcer les connexions
            important_concepts = await self.memory.search_semantic(['concept', 'pattern'], limit=5)
            for item in important_concepts:
                item.importance = min(1.0, item.importance + 0.05)

    async def _graph_evolution(self):
        """Évolution du graphe conceptuel"""
        if self.cycle_count % 15 == 0:  # Tous les 15 cycles
            # Détecte les patterns
            patterns = await self.graph.detect_patterns()
            if patterns:
                logger.info(f"🔍 Patterns detected: {list(patterns.keys())}")

            # Promeut l'imagination
            promoted = await self.graph.promote_imagination(threshold=0.5)
            if promoted:
                logger.info(f"✨ Promoted from imagination: {promoted}")

            # Décroissance de l'imagination
            await self.graph.decay_imagination()

    async def _update_metrics(self):
        """Met à jour les métriques globales"""
        self.total_messages = len(self.bus.history)

    async def run(self, duration: int = 60, cycle_interval: float = 1.0):
        """Lance l'exécution principale"""
        if not self.start_time:
            await self.initialize()

        self.running = True
        end_time = time.time() + duration

        logger.info(f"🏃 Running GrandNexus Mini for {duration} seconds...")

        try:
            while self.running and time.time() < end_time:
                await self.run_cycle()
                await asyncio.sleep(cycle_interval)

                # Affichage périodique des stats
                if self.cycle_count % 30 == 0:
                    await self._print_status()

        except KeyboardInterrupt:
            logger.info("⏹️ Interrupted by user")
        except Exception as e:
            logger.error(f"💥 System error: {e}")
        finally:
            await self.shutdown()

    async def _print_status(self):
        """Affiche le statut du système"""
        memory_stats = self.memory.get_stats()
        graph_stats = self.graph.get_stats()
        core_stats = self.cognitive_core.get_stats()
        env_stats = self.environment.get_stats()

        avg_cycle_time = np.mean(self.performance_history[-10:]) if self.performance_history else 0

        logger.info(f"""
📊 GRANDNEXUS MINI STATUS (Cycle #{self.cycle_count})
🧠 Consciousness: {core_stats['consciousness_level']:.2f} | Load: {core_stats['cognitive_load']:.2f}
🎯 Goals: {core_stats['goals_progress']}
💾 Memory: Hot={memory_stats['hot_cache_size']} | Compressed={memory_stats['compressed_size']} | Abstract={memory_stats['abstract_size']}
🕸️ Graph: Real={graph_stats['real_nodes']} | Imagination={graph_stats['imagination_nodes']} | Patterns={graph_stats['patterns_detected']}
🌍 Environment: Intensity={env_stats['stimulus_intensity']:.2f} | Complexity={env_stats['environmental_state']['complexity']:.2f}
⚡ Performance: {avg_cycle_time:.3f}s/cycle | Messages={self.total_messages}
        """)

    async def shutdown(self):
        """Arrêt propre du système"""
        logger.info("🛑 Shutting down GrandNexus Mini...")

        self.running = False

        # Sauvegarde finale
        await self._save_final_state()

        # Arrêt des composants
        if self.environment:
            await self.environment.stop()

        if self.bus:
            await self.bus.stop()

        logger.info("✅ Shutdown complete")

    async def _save_final_state(self):
        """Sauvegarde l'état final"""
        try:
            final_state = {
                'system_info': {
                    'cycles_completed': self.cycle_count,
                    'total_runtime': time.time() - self.start_time if self.start_time else 0,
                    'total_messages': self.total_messages,
                    'avg_cycle_time': np.mean(self.performance_history) if self.performance_history else 0
                },
                'memory_stats': self.memory.get_stats(),
                'graph_stats': self.graph.get_stats(),
                'cognitive_stats': self.cognitive_core.get_stats(),
                'environment_stats': self.environment.get_stats()
            }

            with open('grandnexus_mini_final_state.json', 'w') as f:
                json.dump(final_state, f, indent=2, default=str)

            logger.info("💾 Final state saved to grandnexus_mini_final_state.json")

        except Exception as e:
            logger.error(f"Error saving final state: {e}")

    def get_comprehensive_stats(self) -> Dict[str, Any]:
        """Statistiques complètes du système"""
        return {
            'orchestrator': {
                'cycle_count': self.cycle_count,
                'total_messages': self.total_messages,
                'running': self.running,
                'uptime': time.time() - self.start_time if self.start_time else 0
            },
            'memory': self.memory.get_stats() if self.memory else {},
            'graph': self.graph.get_stats() if self.graph else {},
            'cognitive_core': self.cognitive_core.get_stats() if self.cognitive_core else {},
            'environment': self.environment.get_stats() if self.environment else {}
        }

# ============================================================
# FONCTION D'ENTRÉE PRINCIPALE POUR COLAB
# ============================================================

async def run_grandnexus_mini(duration: int = 120, cycle_interval: float = 1.0, verbose: bool = True):
    """
    Lance GrandNexus Mini Enhanced - Optimisé pour Colab

    Args:
        duration: Durée d'exécution en secondes
        cycle_interval: Intervalle entre les cycles en secondes
        verbose: Affichage détaillé

    Returns:
        GrandNexusMiniOrchestrator: Instance du système pour inspection
    """

    if verbose:
        print("🌟 GRANDNEXUS MINI ENHANCED - VERSION COLAB")
        print("=" * 50)
        print(f"Duration: {duration}s | Cycle interval: {cycle_interval}s")
        print("=" * 50)

    orchestrator = GrandNexusMiniOrchestrator()

    try:
        await orchestrator.run(duration=duration, cycle_interval=cycle_interval)

        if verbose:
            print("\n" + "=" * 50)
            print("🎉 EXECUTION COMPLETED")
            final_stats = orchestrator.get_comprehensive_stats()
            print(f"Cycles: {final_stats['orchestrator']['cycle_count']}")
            print(f"Messages: {final_stats['orchestrator']['total_messages']}")
            print(f"Uptime: {final_stats['orchestrator']['uptime']:.1f}s")
            print("=" * 50)

        return orchestrator

    except Exception as e:
        logger.error(f"Execution failed: {e}")
        if verbose:
            print(f"❌ Error: {e}")
        return orchestrator

# ============================================================
# FONCTIONS UTILITAIRES POUR COLAB
# ============================================================

def quick_test():
    """Test rapide pour vérifier que tout fonctionne"""
    import asyncio

    async def test():
        print("🧪 Quick test of GrandNexus Mini Enhanced...")
        orchestrator = await run_grandnexus_mini(duration=30, cycle_interval=0.5, verbose=True)
        return orchestrator

    return asyncio.run(test())

def demo_run():
    """Démonstration complète"""
    import asyncio

    async def demo():
        print("🎭 GrandNexus Mini Enhanced - DEMO")
        orchestrator = await run_grandnexus_mini(duration=90, cycle_interval=1.0, verbose=True)

        # Analyse post-exécution
        print("\n📋 POST-EXECUTION ANALYSIS")
        stats = orchestrator.get_comprehensive_stats()

        print(f"🧠 Consciousness Level: {stats['cognitive_core'].get('consciousness_level', 0):.2f}")
        print(f"💾 Memory Items: {stats['memory'].get('hot_cache_size', 0) + stats['memory'].get('compressed_size', 0)}")
        print(f"🕸️ Graph Nodes: {stats['graph'].get('real_nodes', 0)} real + {stats['graph'].get('imagination_nodes', 0)} imagination")
        print(f"🎯 Goal Progress: {stats['cognitive_core'].get('goals_progress', {})}")

        return orchestrator

    return asyncio.run(demo())

# Usage pour Colab:
# 1. Test rapide: orchestrator = quick_test()
# 2. Démo complète: orchestrator = demo_run()
# 3. Custom: orchestrator = asyncio.run(run_grandnexus_mini(duration=60))

Exception ignored in: <coroutine object test_fixed_core at 0x7fd4e84313c0>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object test_fixed_core at 0x7fd4e84313c0>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


In [ ]:
# ============================================================
# SYSTÈME DE RÉGULATION DE CHARGE COGNITIVE
# À intégrer dans AdaptiveCognitiveCore
# ============================================================

import asyncio
import logging
from typing import Dict, Any, List
from collections import deque
import numpy as np

logger = logging.getLogger("GrandNexus.CognitiveRegulator")

class CognitiveLoadRegulator:
    """
    Système de régulation proportionnelle de la charge cognitive
    Maintient le système dans une zone de performance optimale
    """

    def __init__(self, cognitive_core):
        self.core = cognitive_core

        # Paramètres de régulation
        self.target_load = 0.7  # Charge optimale
        self.max_safe_load = 1.2  # Seuil critique
        self.min_load = 0.3  # Seuil de sous-utilisation

        # Vitesses d'adaptation
        self.aggressive_reduction_rate = 0.4  # Réduction rapide en urgence
        self.gentle_reduction_rate = 0.1  # Réduction progressive
        self.expansion_rate = 0.05  # Augmentation prudente

        # Historique pour analyse de tendance
        self.load_history = deque(maxlen=20)
        self.regulation_history = []

        # Compteurs de régulation
        self.total_regulations = 0
        self.emergency_regulations = 0
        self.gentle_regulations = 0

        # État
        self.last_regulation_time = 0
        self.regulation_cooldown = 2.0  # secondes entre régulations

    async def regulate(self) -> Dict[str, Any]:
        """
        Effectue un cycle de régulation
        Returns: Rapport de régulation
        """
        import time
        current_time = time.time()

        # Vérifie le cooldown
        if current_time - self.last_regulation_time < self.regulation_cooldown:
            return {'action': 'cooldown', 'reason': 'too_soon'}

        current_load = self.core.cognitive_load
        self.load_history.append(current_load)

        regulation_report = {
            'timestamp': current_time,
            'load_before': current_load,
            'load_target': self.target_load,
            'action': 'none',
            'interventions': []
        }

        # Analyse de la tendance
        load_trend = self._calculate_load_trend()

        # Décision de régulation basée sur le niveau et la tendance
        if current_load > self.max_safe_load:
            # URGENCE : Réduction agressive
            await self._emergency_reduction(regulation_report)
            self.emergency_regulations += 1

        elif current_load > self.target_load:
            # Charge élevée : Réduction douce
            if load_trend > 0:  # Et la charge augmente encore
                await self._gentle_reduction(regulation_report)
                self.gentle_regulations += 1
            else:
                regulation_report['action'] = 'monitor'
                regulation_report['reason'] = 'load_high_but_decreasing'

        elif current_load < self.min_load:
            # Sous-utilisation : Augmentation de complexité
            await self._increase_complexity(regulation_report)

        else:
            # Zone optimale
            regulation_report['action'] = 'optimal'
            regulation_report['reason'] = 'load_in_target_range'

        # Enregistrement
        regulation_report['load_after'] = self.core.cognitive_load
        regulation_report['load_change'] = regulation_report['load_after'] - regulation_report['load_before']
        regulation_report['trend'] = load_trend

        self.regulation_history.append(regulation_report)
        self.total_regulations += 1
        self.last_regulation_time = current_time

        # Log si action prise
        if regulation_report['action'] not in ['none', 'optimal', 'monitor', 'cooldown']:
            logger.info(f"🔧 Regulation: {regulation_report['action']} | "
                       f"Load: {regulation_report['load_before']:.2f} → {regulation_report['load_after']:.2f}")

        return regulation_report

    def _calculate_load_trend(self) -> float:
        """Calcule la tendance de la charge (montée/descente)"""
        if len(self.load_history) < 3:
            return 0.0

        recent = list(self.load_history)[-5:]

        # Régression linéaire simple
        n = len(recent)
        x = list(range(n))
        mean_x = sum(x) / n
        mean_y = sum(recent) / n

        numerator = sum((x[i] - mean_x) * (recent[i] - mean_y) for i in range(n))
        denominator = sum((x[i] - mean_x) ** 2 for i in range(n))

        if denominator == 0:
            return 0.0

        slope = numerator / denominator
        return slope

    async def _emergency_reduction(self, report: Dict[str, Any]):
        """Réduction agressive en situation d'urgence"""
        report['action'] = 'emergency_reduction'
        report['severity'] = 'critical'
        interventions = []

        # 1. Réduction drastique des stratégies actives
        if hasattr(self.core, 'active_strategies'):
            original_count = len(self.core.active_strategies)
            if original_count > 1:
                # Ne garde que la stratégie la plus efficace
                best_strategy = max(
                    self.core.active_strategies,
                    key=lambda s: self.core.strategy_effectiveness.get(s, 0)
                ) if self.core.active_strategies else None

                self.core.active_strategies = {best_strategy} if best_strategy else set()
                interventions.append({
                    'type': 'strategy_reduction',
                    'from': original_count,
                    'to': len(self.core.active_strategies)
                })

        # 2. Compression agressive de la mémoire
        if 'memory' in self.core.modules:
            memory = self.core.modules['memory']
            if hasattr(memory, 'hot_cache'):
                original_size = len(memory.hot_cache)

                # Force l'éviction des items les moins importants
                items_to_keep = int(original_size * (1 - self.aggressive_reduction_rate))

                # Trie par importance et garde les meilleurs
                sorted_items = sorted(
                    memory.hot_cache.items(),
                    key=lambda x: getattr(x[1], 'importance', 0) * getattr(x[1], 'access_count', 0),
                    reverse=True
                )

                # Reconstruit le cache avec seulement les items importants
                memory.hot_cache = dict(sorted_items[:items_to_keep])

                interventions.append({
                    'type': 'memory_compression',
                    'from': original_size,
                    'to': len(memory.hot_cache),
                    'freed': original_size - len(memory.hot_cache)
                })

        # 3. Élagage du graphe conceptuel
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']
            if hasattr(graph, 'imagination_nodes'):
                original_count = len(graph.imagination_nodes)

                # Supprime tous les concepts imagination avec faible confiance
                to_remove = [
                    name for name, node in graph.imagination_nodes.items()
                    if getattr(node, 'confidence', 0) < 0.3
                ]

                for name in to_remove:
                    del graph.imagination_nodes[name]

                interventions.append({
                    'type': 'imagination_pruning',
                    'removed': len(to_remove),
                    'remaining': len(graph.imagination_nodes)
                })

        # 4. Reset des compteurs d'erreur pour éviter la spirale
        if hasattr(self.core, 'error_count'):
            old_errors = self.core.error_count
            self.core.error_count = int(self.core.error_count * 0.5)
            interventions.append({
                'type': 'error_reset',
                'from': old_errors,
                'to': self.core.error_count
            })

        # 5. Réduction du taux d'adaptation pour stabiliser
        if hasattr(self.core, 'adaptation_rate'):
            old_rate = self.core.adaptation_rate
            self.core.adaptation_rate = max(0.05, self.core.adaptation_rate * 0.7)
            interventions.append({
                'type': 'adaptation_slowdown',
                'from': old_rate,
                'to': self.core.adaptation_rate
            })

        report['interventions'] = interventions
        logger.warning(f"⚠️ EMERGENCY REDUCTION: {len(interventions)} interventions applied")

    async def _gentle_reduction(self, report: Dict[str, Any]):
        """Réduction douce et progressive"""
        report['action'] = 'gentle_reduction'
        report['severity'] = 'moderate'
        interventions = []

        # 1. Réduction progressive des stratégies
        if hasattr(self.core, 'active_strategies') and len(self.core.active_strategies) > 2:
            # Enlève la stratégie la moins efficace
            worst_strategy = min(
                self.core.active_strategies,
                key=lambda s: self.core.strategy_effectiveness.get(s, 0.5)
            )
            self.core.active_strategies.discard(worst_strategy)
            interventions.append({
                'type': 'strategy_removal',
                'removed': worst_strategy,
                'remaining': len(self.core.active_strategies)
            })

        # 2. Consolidation de mémoire douce
        if 'memory' in self.core.modules:
            memory = self.core.modules['memory']
            if hasattr(memory, 'hot_cache') and len(memory.hot_cache) > 20:
                # Éviction des 20% items les moins utilisés
                items_to_evict = int(len(memory.hot_cache) * 0.2)

                sorted_items = sorted(
                    memory.hot_cache.items(),
                    key=lambda x: getattr(x[1], 'access_count', 0)
                )

                # Éviction
                for key, _ in sorted_items[:items_to_evict]:
                    memory.hot_cache.pop(key, None)

                interventions.append({
                    'type': 'memory_eviction',
                    'evicted': items_to_evict,
                    'remaining': len(memory.hot_cache)
                })

        # 3. Decay de l'imagination
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']
            if hasattr(graph, 'decay_imagination'):
                await graph.decay_imagination()
                interventions.append({
                    'type': 'imagination_decay',
                    'remaining': len(getattr(graph, 'imagination_nodes', {}))
                })

        report['interventions'] = interventions

    async def _increase_complexity(self, report: Dict[str, Any]):
        """Augmente la complexité quand le système est sous-utilisé"""
        report['action'] = 'complexity_increase'
        report['severity'] = 'low'
        interventions = []

        # 1. Activation d'une nouvelle stratégie
        if hasattr(self.core, 'strategy_effectiveness'):
            available_strategies = [
                s for s in self.core.strategy_effectiveness.keys()
                if s not in getattr(self.core, 'active_strategies', set())
            ]

            if available_strategies and len(self.core.active_strategies) < 4:
                # Choisit une stratégie avec bonne efficacité mais pas utilisée
                new_strategy = max(
                    available_strategies,
                    key=lambda s: self.core.strategy_effectiveness.get(s, 0.5)
                )
                self.core.active_strategies.add(new_strategy)
                interventions.append({
                    'type': 'strategy_activation',
                    'added': new_strategy,
                    'total': len(self.core.active_strategies)
                })

        # 2. Stimulation de l'imagination
        if 'graph' in self.core.modules:
            graph = self.core.modules['graph']

            # Crée quelques concepts imaginaires pour stimuler l'exploration
            import time
            for i in range(2):
                concept_name = f"explore_{int(time.time())}_{i}"
                concept_data = {
                    'type': 'exploratory',
                    'confidence': 0.4,
                    'source': 'complexity_increase'
                }
                if hasattr(graph, 'add_concept'):
                    await graph.add_concept(concept_name, concept_data, plane='imagination')

            interventions.append({
                'type': 'imagination_stimulation',
                'added_concepts': 2
            })

        # 3. Augmentation douce du taux d'adaptation
        if hasattr(self.core, 'adaptation_rate'):
            old_rate = self.core.adaptation_rate
            self.core.adaptation_rate = min(0.3, self.core.adaptation_rate * 1.1)
            interventions.append({
                'type': 'adaptation_acceleration',
                'from': old_rate,
                'to': self.core.adaptation_rate
            })

        report['interventions'] = interventions

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du régulateur"""
        return {
            'total_regulations': self.total_regulations,
            'emergency_regulations': self.emergency_regulations,
            'gentle_regulations': self.gentle_regulations,
            'current_load': self.core.cognitive_load,
            'target_load': self.target_load,
            'load_trend': self._calculate_load_trend(),
            'avg_load': np.mean(list(self.load_history)) if self.load_history else 0,
            'load_stability': np.std(list(self.load_history)) if len(self.load_history) > 1 else 0
        }



# ============================================================
# FONCTION DE TEST DU RÉGULATEUR
# ============================================================

async def test_regulator():
    """Test du système de régulation"""
    from grandnexus_mini_enhanced_part3 import GrandNexusMiniOrchestrator

    print("🧪 Testing Cognitive Load Regulator...")

    orchestrator = GrandNexusMiniOrchestrator()
    await orchestrator.initialize()

    # Active la régulation
    if hasattr(orchestrator.cognitive_core, 'load_regulator'):
        regulator = orchestrator.cognitive_core.load_regulator

        # Simule différents niveaux de charge
        test_scenarios = [
            (0.5, "Normal load"),
            (0.9, "High load"),
            (1.5, "Critical load"),
            (0.2, "Under-utilized")
        ]

        for test_load, description in test_scenarios:
            print(f"\n{'='*60}")
            print(f"Scenario: {description} (Load: {test_load})")
            print(f"{'='*60}")

            # Force la charge
            orchestrator.cognitive_core.cognitive_load = test_load

            # Régulation
            report = await regulator.regulate()

            # Affiche le résultat
            print(f"Action: {report['action']}")
            print(f"Load: {report['load_before']:.2f} → {report['load_after']:.2f}")
            if report.get('interventions'):
                print(f"Interventions: {len(report['interventions'])}")
                for intervention in report['interventions']:
                    print(f"  • {intervention['type']}")

            await asyncio.sleep(3)  # Cooldown

        # Stats finales
        print(f"\n{'='*60}")
        print("REGULATOR STATS")
        print(f"{'='*60}")
        stats = regulator.get_stats()
        for key, value in stats.items():
            if isinstance(value, float):
                print(f"{key}: {value:.3f}")
            else:
                print(f"{key}: {value}")

    await orchestrator.shutdown()
    print("\n✅ Test complete")


print("✅ Cognitive Load Regulator loaded!")
print("\nUsage:")
print("1. Le régulateur est automatiquement intégré dans AdaptiveCognitiveCore")
print("2. Il s'active à chaque cognitive_cycle()")
print("3. Test: await test_regulator()")

✅ Cognitive Load Regulator loaded!

Usage:
1. Le régulateur est automatiquement intégré dans AdaptiveCognitiveCore
2. Il s'active à chaque cognitive_cycle()
3. Test: await test_regulator()


In [ ]:
# ============================================================
# SYSTÈME DE CONSOLIDATION DES CONNAISSANCES
# Résout le problème d'accumulation sans organisation
# ============================================================

import asyncio
import logging
from typing import Dict, Any, List, Set, Tuple
from collections import defaultdict, Counter
import time
import numpy as np

logger = logging.getLogger("GrandNexus.Consolidation")

class KnowledgeConsolidator:
    """
    Système de consolidation périodique qui empêche l'accumulation chaotique
    Inspiré des principes de consolidation de la mémoire biologique
    """

    def __init__(self, orchestrator):
        self.orchestrator = orchestrator
        self.memory = orchestrator.memory
        self.graph = orchestrator.graph
        self.cognitive_core = orchestrator.cognitive_core

        # Configuration de consolidation
        self.consolidation_interval = 100  # Cycles entre consolidations
        self.last_consolidation = 0

        # Seuils (AJUSTÉS - moins agressifs)
        self.similarity_threshold = 0.8  # Pour fusion de concepts (était 0.6 - TROP BAS)
        self.weak_edge_threshold = 0.15  # Pour élagage connexions (abaissé de 0.2)
        self.archive_age = 300  # Secondes (5 minutes)
        self.min_access_count = 2  # Minimum accès pour garder en hot

        # Protections (NOUVEAU)
        self.min_concepts = 30  # Minimum de concepts à garder
        self.min_memory_items = 15  # Minimum d'items mémoire
        self.max_merge_per_cycle = 2  # Max fusions par consolidation (était 3)

        # Métriques
        self.consolidations_performed = 0
        self.concepts_merged = 0
        self.edges_pruned = 0
        self.items_archived = 0
        self.consolidation_history = []

    async def should_consolidate(self) -> bool:
        """Détermine si une consolidation est nécessaire"""
        cycles_since_last = self.orchestrator.cycle_count - self.last_consolidation

        # PROTECTION: Cooldown minimum de 50 cycles (nouveau)
        if cycles_since_last < 50:
            return False

        # Consolidation périodique
        if cycles_since_last >= self.consolidation_interval:
            return True

        # Consolidation d'urgence si surcharge (mais max toutes les 50 cycles)
        if cycles_since_last >= 50:  # Ajouté
            if hasattr(self.cognitive_core, 'cognitive_load'):
                if self.cognitive_core.cognitive_load > 1.8:  # Augmenté de 1.5
                    logger.info("🚨 Emergency consolidation triggered (high load)")
                    return True

            # Consolidation si mémoire saturée
            if hasattr(self.memory, 'hot_cache'):
                if len(self.memory.hot_cache) > 60:  # Augmenté de 50
                    logger.info("🚨 Emergency consolidation triggered (memory full)")
                    return True

        return False

    async def consolidate(self) -> Dict[str, Any]:
        """
        Effectue une consolidation complète
        Returns: Rapport de consolidation
        """
        consolidation_start = time.time()

        report = {
            'timestamp': consolidation_start,
            'cycle': self.orchestrator.cycle_count,
            'phases': {},
            'metrics_before': self._capture_metrics(),
            'improvements': {}
        }

        logger.info("🔄 Starting knowledge consolidation...")

        # Phase 1: Consolidation de la mémoire
        memory_report = await self._consolidate_memory()
        report['phases']['memory'] = memory_report

        # Phase 2: Consolidation du graphe conceptuel
        graph_report = await self._consolidate_graph()
        report['phases']['graph'] = graph_report

        # Phase 3: Optimisation des stratégies cognitives
        strategy_report = await self._optimize_strategies()
        report['phases']['strategies'] = strategy_report

        # Phase 4: Calcul des améliorations
        report['metrics_after'] = self._capture_metrics()
        report['improvements'] = self._calculate_improvements(
            report['metrics_before'],
            report['metrics_after']
        )

        # Finalisation
        self.last_consolidation = self.orchestrator.cycle_count
        self.consolidations_performed += 1

        consolidation_time = time.time() - consolidation_start
        report['duration'] = consolidation_time

        self.consolidation_history.append(report)
        if len(self.consolidation_history) > 50:
            self.consolidation_history = self.consolidation_history[-25:]

        logger.info(f"✅ Consolidation complete in {consolidation_time:.2f}s")
        self._log_improvements(report['improvements'])

        return report

    async def _consolidate_memory(self) -> Dict[str, Any]:
        """Phase 1: Consolidation de la mémoire hiérarchique"""
        report = {
            'archived': 0,
            'compressed': 0,
            'forgotten': 0
        }

        if not hasattr(self.memory, 'hot_cache'):
            return report

        current_time = time.time()
        to_archive = []
        to_forget = []

        # 1. Archive les items anciens peu accédés
        for key, item in list(self.memory.hot_cache.items()):
            age = current_time - getattr(item, 'timestamp', current_time)
            access_count = getattr(item, 'access_count', 0)
            importance = getattr(item, 'importance', 0.5)

            # Items anciens ET peu importants → oubli
            if age > self.archive_age * 2 and access_count < 2 and importance < 0.3:
                to_forget.append(key)

            # Items anciens mais potentiellement importants → compression
            elif age > self.archive_age and access_count < self.min_access_count:
                to_archive.append(key)

        # Exécution des archivages
        # PROTECTION: Ne pas archiver si mémoire déjà petite
        if len(self.memory.hot_cache) > self.min_memory_items:
            for key in to_archive:
                item = self.memory.hot_cache.pop(key, None)
                if item and hasattr(self.memory, 'compressed_store'):
                    # Compresse avant d'archiver
                    compressed_value = self._compress_item_value(getattr(item, 'value', item))
                    item.value = compressed_value
                    self.memory.compressed_store[key] = item
                    report['archived'] += 1
        else:
            logger.warning(f"⚠️ Memory at minimum ({len(self.memory.hot_cache)}), skipping archive")

        # Exécution des oublis
        for key in to_forget:
            self.memory.hot_cache.pop(key, None)
            report['forgotten'] += 1

        # 2. Compression sémantique des items similaires en compressed_store
        if hasattr(self.memory, 'compressed_store'):
            merged = await self._merge_similar_memories()
            report['compressed'] = merged

        self.items_archived += report['archived']

        return report

    def _compress_item_value(self, value: Any) -> Any:
        """Compresse une valeur en gardant l'essence"""
        if isinstance(value, str):
            # Garde les 5 premiers mots-clés
            words = value.split()
            important_words = [w for w in words if len(w) > 3][:5]
            return " ".join(important_words)

        elif isinstance(value, dict):
            # Garde seulement les clés essentielles
            essential_keys = ['type', 'name', 'id', 'result', 'value']
            return {k: v for k, v in value.items() if k in essential_keys}

        elif isinstance(value, (list, tuple)):
            # Garde max 5 éléments
            return value[:5] if len(value) > 5 else value

        return value

    async def _merge_similar_memories(self) -> int:
        """Fusionne les souvenirs similaires en compressed_store"""
        if not hasattr(self.memory, 'compressed_store'):
            return 0

        merged_count = 0
        items = list(self.memory.compressed_store.items())

        # Compare les items par paires
        to_merge = []
        for i, (key1, item1) in enumerate(items):
            for key2, item2 in items[i+1:]:
                similarity = self._calculate_memory_similarity(item1, item2)
                if similarity > 0.7:  # Très similaires
                    to_merge.append((key1, key2, similarity))

        # Fusionne les paires identifiées
        for key1, key2, similarity in to_merge[:5]:  # Max 5 fusions par consolidation
            if key1 in self.memory.compressed_store and key2 in self.memory.compressed_store:
                item1 = self.memory.compressed_store[key1]
                item2 = self.memory.compressed_store[key2]

                # Crée un item fusionné
                merged_item = self._merge_memory_items(item1, item2)

                # Garde le plus ancien key, supprime l'autre
                self.memory.compressed_store[key1] = merged_item
                del self.memory.compressed_store[key2]
                merged_count += 1

        return merged_count

    def _calculate_memory_similarity(self, item1, item2) -> float:
        """Calcule la similarité entre deux items mémoire"""
        # Similarité basée sur les tags sémantiques
        tags1 = set(getattr(item1, 'semantic_tags', []))
        tags2 = set(getattr(item2, 'semantic_tags', []))

        if not tags1 or not tags2:
            return 0.0

        intersection = len(tags1 & tags2)
        union = len(tags1 | tags2)

        return intersection / union if union > 0 else 0.0

    def _merge_memory_items(self, item1, item2):
        """Fusionne deux items mémoire"""
        # Garde l'item le plus important
        if getattr(item1, 'importance', 0) >= getattr(item2, 'importance', 0):
            merged = item1
        else:
            merged = item2

        # Combine les tags sémantiques
        tags1 = set(getattr(item1, 'semantic_tags', []))
        tags2 = set(getattr(item2, 'semantic_tags', []))
        merged.semantic_tags = list(tags1 | tags2)

        # Moyenne de l'importance
        merged.importance = (getattr(item1, 'importance', 0.5) +
                           getattr(item2, 'importance', 0.5)) / 2

        # Somme des accès
        merged.access_count = (getattr(item1, 'access_count', 0) +
                              getattr(item2, 'access_count', 0))

        return merged

    async def _consolidate_graph(self) -> Dict[str, Any]:
        """Phase 2: Consolidation du graphe conceptuel"""
        report = {
            'concepts_merged': 0,
            'edges_pruned': 0,
            'imagination_consolidated': 0
        }

        if not hasattr(self.graph, 'nodes'):
            return report

        # 1. Fusionne les concepts similaires
        merged = await self._merge_similar_concepts()
        report['concepts_merged'] = merged

        # 2. Élague les connexions faibles
        pruned = await self._prune_weak_edges()
        report['edges_pruned'] = pruned

        # 3. Consolide l'imagination
        if hasattr(self.graph, 'imagination_nodes'):
            consolidated = await self._consolidate_imagination()
            report['imagination_consolidated'] = consolidated

        self.concepts_merged += report['concepts_merged']
        self.edges_pruned += report['edges_pruned']

        return report

    async def _merge_similar_concepts(self) -> int:
        """Fusionne les concepts très similaires dans le graphe"""
        if not hasattr(self.graph, 'nodes'):
            return 0

        merged_count = 0
        nodes_list = list(self.graph.nodes.items())
        to_merge = []

        # Identifie les paires à fusionner
        for i, (name1, node1) in enumerate(nodes_list):
            for name2, node2 in nodes_list[i+1:]:
                similarity = self._calculate_concept_similarity(name1, node1, name2, node2)
                if similarity > self.similarity_threshold:
                    to_merge.append((name1, name2, similarity))

        # Fusionne max 2 paires par consolidation (pour ne pas trop perturber)
        # PROTECTION: Ne pas descendre sous le minimum
        if len(self.graph.nodes) <= self.min_concepts:
            logger.warning(f"⚠️ Graph at minimum size ({len(self.graph.nodes)}), skipping merge")
            return 0

        for name1, name2, similarity in sorted(to_merge, key=lambda x: x[2], reverse=True)[:self.max_merge_per_cycle]:
            if name1 in self.graph.nodes and name2 in self.graph.nodes:
                await self._merge_two_concepts(name1, name2)
                merged_count += 1

        return merged_count

    def _calculate_concept_similarity(self, name1: str, node1, name2: str, node2) -> float:
        """Calcule la similarité entre deux concepts"""
        similarity = 0.0

        # Similarité des noms (Levenshtein simplifié)
        name_sim = self._string_similarity(name1.lower(), name2.lower())
        similarity += name_sim * 0.3

        # Similarité des données
        data1 = getattr(node1, 'data', {})
        data2 = getattr(node2, 'data', {})

        if data1.get('type') == data2.get('type'):
            similarity += 0.3

        # Similarité des connexions (voisins communs)
        connections1 = set(getattr(node1, 'connections', {}).keys())
        connections2 = set(getattr(node2, 'connections', {}).keys())

        if connections1 or connections2:
            common = len(connections1 & connections2)
            total = len(connections1 | connections2)
            connection_sim = common / total if total > 0 else 0
            similarity += connection_sim * 0.4

        return similarity

    def _string_similarity(self, s1: str, s2: str) -> float:
        """Similarité simple entre deux chaînes"""
        # Jaccard sur les caractères
        set1 = set(s1)
        set2 = set(s2)

        if not set1 or not set2:
            return 0.0

        intersection = len(set1 & set2)
        union = len(set1 | set2)

        return intersection / union

    async def _merge_two_concepts(self, name1: str, name2: str):
        """Fusionne deux concepts en un seul"""
        node1 = self.graph.nodes[name1]
        node2 = self.graph.nodes[name2]

        # Garde le plus ancien (ou celui avec plus de connexions)
        if len(getattr(node1, 'connections', {})) >= len(getattr(node2, 'connections', {})):
            keeper = name1
            removed = name2
            keeper_node = node1
            removed_node = node2
        else:
            keeper = name2
            removed = name1
            keeper_node = node2
            removed_node = node1

        # Transfère les connexions
        for neighbor, strength in getattr(removed_node, 'connections', {}).items():
            if neighbor != keeper:
                # Ajoute ou renforce la connexion
                current_strength = keeper_node.connections.get(neighbor, 0)
                keeper_node.connections[neighbor] = max(current_strength, strength)

                # Met à jour la connexion inverse
                if neighbor in self.graph.nodes:
                    neighbor_node = self.graph.nodes[neighbor]
                    if hasattr(neighbor_node, 'connections'):
                        neighbor_node.connections[keeper] = keeper_node.connections[neighbor]
                        neighbor_node.connections.pop(removed, None)

        # Supprime le concept fusionné
        del self.graph.nodes[removed]

        logger.debug(f"Merged concepts: {removed} → {keeper}")

    async def _prune_weak_edges(self) -> int:
        """Élague les connexions faibles dans le graphe"""
        if not hasattr(self.graph, 'nodes'):
            return 0

        pruned = 0

        for node_name, node in self.graph.nodes.items():
            if not hasattr(node, 'connections'):
                continue

            # Identifie les connexions faibles
            weak_connections = [
                neighbor for neighbor, strength in node.connections.items()
                if strength < self.weak_edge_threshold
            ]

            # MODIFIÉ: Supprime max 30% des connexions faibles (était 50%)
            to_remove = weak_connections[:len(weak_connections)//3]

            for neighbor in to_remove:
                node.connections.pop(neighbor, None)
                pruned += 1

                # Supprime aussi la connexion inverse
                if neighbor in self.graph.nodes:
                    neighbor_node = self.graph.nodes[neighbor]
                    if hasattr(neighbor_node, 'connections'):
                        neighbor_node.connections.pop(node_name, None)

        return pruned

    async def _consolidate_imagination(self) -> int:
        """Consolide les concepts d'imagination"""
        if not hasattr(self.graph, 'imagination_nodes'):
            return 0

        consolidated = 0
        current_time = time.time()

        # Identifie les concepts imagination à promouvoir ou supprimer
        to_promote = []
        to_remove = []

        for name, node in list(self.graph.imagination_nodes.items()):
            age = current_time - getattr(node, 'creation_time', current_time)
            confidence = getattr(node, 'confidence', 0)
            access_count = getattr(node, 'access_count', 0)

            # Promeut si forte confiance ET utilisé
            if confidence > 0.6 and access_count > 3:
                to_promote.append(name)

            # Supprime si vieux ET faible confiance
            elif age > 60 and confidence < 0.3:  # 1 minute
                to_remove.append(name)

        # Promotion
        for name in to_promote:
            node = self.graph.imagination_nodes.pop(name, None)
            if node:
                node.plane = 'consolidated'
                self.graph.nodes[name] = node
                consolidated += 1
                logger.debug(f"Promoted from imagination: {name}")

        # Suppression
        for name in to_remove:
            self.graph.imagination_nodes.pop(name, None)

        return consolidated

    async def _optimize_strategies(self) -> Dict[str, Any]:
        """Phase 3: Optimisation des stratégies cognitives"""
        report = {
            'strategies_pruned': 0,
            'effectiveness_updated': 0
        }

        if not hasattr(self.cognitive_core, 'strategy_effectiveness'):
            return report

        # 1. Supprime les stratégies avec efficacité très faible
        low_performers = [
            strategy for strategy, effectiveness in self.cognitive_core.strategy_effectiveness.items()
            if effectiveness < 0.2
        ]

        for strategy in low_performers[:2]:  # Max 2 par consolidation
            self.cognitive_core.strategy_effectiveness.pop(strategy, None)
            report['strategies_pruned'] += 1

        # 2. Decay progressif de l'efficacité pour forcer réévaluation
        for strategy in self.cognitive_core.strategy_effectiveness:
            old_value = self.cognitive_core.strategy_effectiveness[strategy]
            new_value = old_value * 0.95  # Decay de 5%
            self.cognitive_core.strategy_effectiveness[strategy] = max(0.1, new_value)
            report['effectiveness_updated'] += 1

        return report

    def _capture_metrics(self) -> Dict[str, Any]:
        """Capture les métriques système"""
        metrics = {}

        # Mémoire
        if hasattr(self.memory, 'hot_cache'):
            metrics['memory_hot'] = len(self.memory.hot_cache)
            metrics['memory_compressed'] = len(getattr(self.memory, 'compressed_store', {}))
            metrics['memory_total'] = metrics['memory_hot'] + metrics['memory_compressed']

        # Graphe
        if hasattr(self.graph, 'nodes'):
            metrics['graph_nodes'] = len(self.graph.nodes)
            metrics['graph_imagination'] = len(getattr(self.graph, 'imagination_nodes', {}))

            total_connections = sum(len(getattr(node, 'connections', {}))
                                  for node in self.graph.nodes.values())
            metrics['graph_connections'] = total_connections

        # Cognitif
        if hasattr(self.cognitive_core, 'cognitive_load'):
            metrics['cognitive_load'] = self.cognitive_core.cognitive_load
            metrics['consciousness'] = getattr(self.cognitive_core, 'consciousness_level', 0)

        return metrics

    def _calculate_improvements(self, before: Dict, after: Dict) -> Dict[str, float]:
        """Calcule les améliorations après consolidation"""
        improvements = {}

        for key in before:
            if key in after:
                change = after[key] - before[key]
                if before[key] != 0:
                    percent_change = (change / abs(before[key])) * 100
                else:
                    percent_change = 0

                improvements[key] = {
                    'before': before[key],
                    'after': after[key],
                    'change': change,
                    'percent': percent_change
                }

        return improvements

    def _log_improvements(self, improvements: Dict):
        """Log les améliorations de façon lisible"""
        for metric, data in improvements.items():
            if abs(data['change']) > 0:
                direction = "↑" if data['change'] > 0 else "↓"
                logger.info(f"  {direction} {metric}: {data['before']} → {data['after']} ({data['change']:+.1f})")

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du consolidateur"""
        return {
            'total_consolidations': self.consolidations_performed,
            'concepts_merged': self.concepts_merged,
            'edges_pruned': self.edges_pruned,
            'items_archived': self.items_archived,
            'last_consolidation_cycle': self.last_consolidation,
            'cycles_since_last': self.orchestrator.cycle_count - self.last_consolidation
        }

print("✅ Knowledge Consolidation System loaded!")
print("\nIntegration:")
print("1. Add to __init__: self.consolidator = KnowledgeConsolidator(self)")
print("2. Add to run_cycle: consolidation check (see comments above)")
print("3. Consolidation runs every 100 cycles or when load > 1.5")

✅ Knowledge Consolidation System loaded!

Integration:
1. Add to __init__: self.consolidator = KnowledgeConsolidator(self)
2. Add to run_cycle: consolidation check (see comments above)
3. Consolidation runs every 100 cycles or when load > 1.5


In [ ]:
# ============================================================
# GRANDNEXUS MONITORING - VERSION STANDALONE POUR COLAB
# Copier-coller ce code APRÈS avoir chargé les parties 1, 2 et 3
# ============================================================

import asyncio
import time
import json
import numpy as np
from collections import defaultdict, deque, Counter
from typing import Dict, List, Any, Optional
from datetime import datetime

class GrandNexusMonitor:
    """Système de monitoring en temps réel pour GrandNexus"""

    def __init__(self):
        self.session_start = time.time()
        self.performance_snapshots = []
        self.anomalies_detected = []
        self.monitoring_active = False
        self.snapshot_interval = 5.0
        self.max_history_size = 200

    def start_monitoring(self, orchestrator):
        """Démarre le monitoring"""
        self.orchestrator = orchestrator
        self.monitoring_active = True
        self.session_start = time.time()
        print("🔍 GrandNexus Monitor STARTED")

    def stop_monitoring(self):
        """Arrête le monitoring"""
        self.monitoring_active = False
        print("🔍 GrandNexus Monitor STOPPED")

    def capture_snapshot(self) -> Dict[str, Any]:
        """Capture un instantané du système"""
        if not hasattr(self, 'orchestrator'):
            return {}

        try:
            orch = self.orchestrator
            snapshot = {
                'timestamp': time.time(),
                'session_time': time.time() - self.session_start,
                'cycle_count': getattr(orch, 'cycle_count', 0),
                'system': {
                    'running': getattr(orch, 'running', False),
                    'cycle_count': getattr(orch, 'cycle_count', 0),
                    'total_messages': getattr(orch, 'total_messages', 0),
                    'uptime': time.time() - getattr(orch, 'start_time', time.time())
                }
            }

            # Métriques cognitives
            if hasattr(orch, 'cognitive_core') and orch.cognitive_core:
                core = orch.cognitive_core
                goals_data = {}
                if hasattr(core, 'goals'):
                    goals_data = {
                        name: {
                            'progress': goal.current_value,
                            'target': goal.target_value,
                            'priority': goal.priority
                        }
                        for name, goal in core.goals.items()
                    }

                snapshot['cognitive'] = {
                    'consciousness_level': getattr(core, 'consciousness_level', 0),
                    'cognitive_load': getattr(core, 'cognitive_load', 0),
                    'adaptation_rate': getattr(core, 'adaptation_rate', 0),
                    'success_rate': core.success_count / max(1, core.success_count + core.error_count) if hasattr(core, 'success_count') else 0,
                    'goals': goals_data
                }

            # Métriques mémoire
            if hasattr(orch, 'memory') and orch.memory:
                memory = orch.memory
                snapshot['memory'] = {
                    'hot_cache_size': len(getattr(memory, 'hot_cache', {})),
                    'compressed_size': len(getattr(memory, 'compressed_store', {})),
                    'total_items': len(getattr(memory, 'hot_cache', {})) + len(getattr(memory, 'compressed_store', {}))
                }

            # Métriques graphe
            if hasattr(orch, 'graph') and orch.graph:
                graph = orch.graph
                snapshot['graph'] = {
                    'real_nodes': len(getattr(graph, 'nodes', {})),
                    'imagination_nodes': len(getattr(graph, 'imagination_nodes', {})),
                    'total_nodes': len(getattr(graph, 'nodes', {})) + len(getattr(graph, 'imagination_nodes', {}))
                }

            # Métriques communication
            if hasattr(orch, 'bus') and orch.bus:
                bus = orch.bus
                snapshot['communication'] = {
                    'total_messages': len(getattr(bus, 'history', []))
                }

            # Stockage
            self.performance_snapshots.append(snapshot)
            if len(self.performance_snapshots) > self.max_history_size:
                self.performance_snapshots = self.performance_snapshots[-self.max_history_size//2:]

            return snapshot

        except Exception as e:
            return {'timestamp': time.time(), 'error': str(e)}

    def generate_status_report(self, detailed: bool = True) -> str:
        """Génère un rapport textuel détaillé"""
        if not self.performance_snapshots:
            return "❌ No performance data available yet"

        latest = self.performance_snapshots[-1]
        session_time = time.time() - self.session_start

        lines = [
            "=" * 80,
            "🔍 GRANDNEXUS MONITORING REPORT",
            "=" * 80,
            f"📅 Session Time: {session_time:.1f}s",
            f"⏰ Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"📊 Snapshots: {len(self.performance_snapshots)}",
            ""
        ]

        # Système
        if 'system' in latest:
            sys = latest['system']
            lines.extend([
                "🖥️  SYSTEM:",
                f"   Running: {'✅' if sys.get('running', False) else '❌'}",
                f"   Cycles: {sys.get('cycle_count', 0):,}",
                f"   Messages: {sys.get('total_messages', 0):,}",
                f"   Uptime: {sys.get('uptime', 0):.1f}s",
                ""
            ])

        # Cognitif
        if 'cognitive' in latest:
            cog = latest['cognitive']
            lines.extend([
                "🧠 COGNITIVE:",
                f"   Consciousness: {cog.get('consciousness_level', 0)}",
                f"   Load: {cog.get('cognitive_load', 0):.2f}",
                f"   Adaptation Rate: {cog.get('adaptation_rate', 0):.3f}",
                f"   Success Rate: {cog.get('success_rate', 0):.1%}",
                ""
            ])

            goals = cog.get('goals', {})
            if goals:
                lines.append("   🎯 GOALS:")
                for name, data in goals.items():
                    progress = data.get('progress', 0)
                    target = data.get('target', 1)
                    pct = (progress / target * 100) if target > 0 else 0
                    lines.append(f"      • {name}: {progress:.2f}/{target:.2f} ({pct:.0f}%)")
                lines.append("")

        # Mémoire
        if 'memory' in latest:
            mem = latest['memory']
            lines.extend([
                "💾 MEMORY:",
                f"   Total Items: {mem.get('total_items', 0):,}",
                f"   Hot Cache: {mem.get('hot_cache_size', 0):,}",
                f"   Compressed: {mem.get('compressed_size', 0):,}",
                ""
            ])

        # Graphe
        if 'graph' in latest:
            graph = latest['graph']
            lines.extend([
                "🕸️  GRAPH:",
                f"   Real Nodes: {graph.get('real_nodes', 0):,}",
                f"   Imagination: {graph.get('imagination_nodes', 0):,}",
                f"   Total: {graph.get('total_nodes', 0):,}",
                ""
            ])

        # Communication
        if 'communication' in latest:
            comm = latest['communication']
            lines.extend([
                "📡 COMMUNICATION:",
                f"   Total Messages: {comm.get('total_messages', 0):,}",
                ""
            ])

        lines.extend([
            "=" * 80,
            "END OF REPORT",
            "=" * 80
        ])

        return "\n".join(lines)

    def generate_compact_status(self) -> str:
        """Status compact sur une ligne"""
        if not self.performance_snapshots:
            return "⏳ Waiting for data..."

        latest = self.performance_snapshots[-1]

        cycle = latest.get('system', {}).get('cycle_count', 0)
        consciousness = latest.get('cognitive', {}).get('consciousness_level', 0)
        load = latest.get('cognitive', {}).get('cognitive_load', 0)
        nodes = latest.get('graph', {}).get('total_nodes', 0)
        messages = latest.get('communication', {}).get('total_messages', 0)

        return f"🟢 C:{cycle} | Mind:{consciousness} | Load:{load:.2f} | Graph:{nodes} | Msg:{messages}"

    def save_report_to_file(self, filename: str = None):
        """Sauvegarde le rapport"""
        if filename is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"grandnexus_report_{timestamp}.txt"

        report = self.generate_status_report(detailed=True)

        try:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(report)
            print(f"📄 Report saved to: {filename}")
            return filename
        except Exception as e:
            print(f"❌ Failed to save: {e}")
            return None


# ============================================================
# FONCTION SIMPLIFIÉE POUR COLAB
# ============================================================

async def run_with_monitoring(orchestrator, duration: int = 60,
                              report_interval: int = 10):
    """
    Lance le monitoring sur un orchestrateur existant

    Args:
        orchestrator: Instance de GrandNexusMiniOrchestrator déjà initialisée
        duration: Durée d'exécution (secondes)
        report_interval: Intervalle entre rapports (secondes)
    """

    print("\n" + "=" * 80)
    print("🔍 GRANDNEXUS LIVE MONITORING")
    print("=" * 80)
    print(f"Duration: {duration}s | Reports every {report_interval}s")
    print("=" * 80 + "\n")

    monitor = GrandNexusMonitor()
    monitor.start_monitoring(orchestrator)

    try:
        start_time = time.time()
        last_report = start_time
        last_compact = start_time
        last_snapshot = start_time

        while time.time() - start_time < duration:
            current = time.time()

            # Cycle
            try:
                await orchestrator.run_cycle()
            except Exception as e:
                print(f"⚠️  Cycle error: {e}")

            # Snapshot périodique
            if current - last_snapshot >= 5:
                monitor.capture_snapshot()
                last_snapshot = current

            # Status compact toutes les 3s
            if current - last_compact >= 3:
                print(monitor.generate_compact_status())
                last_compact = current

            # Rapport détaillé
            if current - last_report >= report_interval:
                print("\n" + "─" * 80)
                print(monitor.generate_status_report())
                print("─" * 80 + "\n")
                last_report = current

            await asyncio.sleep(0.1)

        # Rapport final
        print("\n" + "=" * 80)
        print("📊 FINAL REPORT")
        print("=" * 80 + "\n")
        print(monitor.generate_status_report())

        # Sauvegarde
        print("\n📁 Saving report...")
        monitor.save_report_to_file()

        return monitor

    except KeyboardInterrupt:
        print("\n⏹️  Interrupted")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        monitor.stop_monitoring()
        print("✅ Monitoring stopped\n")
        return monitor


# ============================================================
# DIAGNOSTIC RAPIDE
# ============================================================

async def quick_snapshot(orchestrator):
    """Prend un snapshot rapide et l'affiche"""
    monitor = GrandNexusMonitor()
    monitor.start_monitoring(orchestrator)
    monitor.capture_snapshot()

    print("\n" + "=" * 80)
    print("🔍 QUICK DIAGNOSTIC SNAPSHOT")
    print("=" * 80)
    print(monitor.generate_status_report())
    print("=" * 80 + "\n")

    return monitor


# ============================================================
# EXEMPLE D'UTILISATION COMPLÈTE
# ============================================================

async def test_monitoring_complete():
    """Test complet avec initialisation et monitoring"""

    print("🚀 Initializing GrandNexus Mini Enhanced...")

    # Création de l'orchestrateur (utilise les classes déjà définies)
    orchestrator = GrandNexusMiniOrchestrator()

    # Initialisation
    await orchestrator.initialize()

    print("✅ Initialization complete\n")

    # Lance avec monitoring
    monitor = await run_with_monitoring(
        orchestrator,
        duration=60,  # 1 minute de test
        report_interval=15
    )

    # Cleanup
    await orchestrator.shutdown()

    return orchestrator, monitor


print("✅ Monitoring system loaded!")
print("\nUsage:")
print("1. Initialiser votre orchestrateur:")
print("   orchestrator = GrandNexusMiniOrchestrator()")
print("   await orchestrator.initialize()")
print("")
print("2. Lancer avec monitoring:")
print("   monitor = await run_with_monitoring(orchestrator, duration=60)")
print("")
print("3. OU test complet:")
print("   orchestrator, monitor = await test_monitoring_complete()")
print("")
print("4. Snapshot rapide:")
print("   await quick_snapshot(orchestrator)")

✅ Monitoring system loaded!

Usage:
1. Initialiser votre orchestrateur:
   orchestrator = GrandNexusMiniOrchestrator()
   await orchestrator.initialize()

2. Lancer avec monitoring:
   monitor = await run_with_monitoring(orchestrator, duration=60)

3. OU test complet:
   orchestrator, monitor = await test_monitoring_complete()

4. Snapshot rapide:
   await quick_snapshot(orchestrator)


In [ ]:
# Méthode manuelle
orchestrator = GrandNexusMiniOrchestrator()
await orchestrator.initialize()
monitor = await run_with_monitoring(orchestrator, duration=60, report_interval=15)


🔍 GRANDNEXUS LIVE MONITORING
Duration: 60s | Reports every 15s

🔍 GrandNexus Monitor STARTED
⏳ Waiting for data...
🟢 C:51 | Mind:0.39725284550381906 | Load:0.18 | Graph:50 | Msg:173
🟢 C:51 | Mind:0.39725284550381906 | Load:0.18 | Graph:50 | Msg:173


🟢 C:100 | Mind:0.43763075313222094 | Load:0.37 | Graph:51 | Msg:363

────────────────────────────────────────────────────────────────────────────────
🔍 GRANDNEXUS MONITORING REPORT
📅 Session Time: 15.1s
⏰ Generated: 2025-10-08 07:57:08
📊 Snapshots: 3

🖥️  SYSTEM:
   Running: ❌
   Cycles: 149
   Messages: 572
   Uptime: 15.1s

🧠 COGNITIVE:
   Consciousness: 0.471567494937278
   Load: 0.60
   Adaptation Rate: 0.100
   Success Rate: 66.4%

   🎯 GOALS:
      • survive: 0.66/1.00 (66%)
      • learn: 0.26/0.80 (32%)
      • adapt: 0.00/0.70 (0%)
      • emerge: 0.80/0.60 (133%)

💾 MEMORY:
   Total Items: 26
   Hot Cache: 9
   Compressed: 17

🕸️  GRAPH:
   Real Nodes: 52
   Imagination: 0
   Total: 52

📡 COMMUNICATION:
   Total Messages: 572

END OF REPORT
────────────────────────────────────────────────────────────────────────────────

🟢 C:149 | Mind:0.471567494937278 | Load:0.60 | Graph:52 | Msg:572
🟢 C:149 | Mind:0.471567494937278 | Load:0.60 | Graph:52 | Msg:572


🟢 C:198 | Mind:0.5006258996878602 | Load:0.80 | Graph:53 | Msg:769
🟢 C:198 | Mind:0.5006258996878602 | Load:0.80 | Graph:53 | Msg:769
🟢 C:247 | Mind:0.5058367881649805 | Load:1.03 | Graph:52 | Msg:989

────────────────────────────────────────────────────────────────────────────────
🔍 GRANDNEXUS MONITORING REPORT
📅 Session Time: 30.2s
⏰ Generated: 2025-10-08 07:57:23
📊 Snapshots: 6

🖥️  SYSTEM:
   Running: ❌
   Cycles: 296
   Messages: 1,147
   Uptime: 30.2s

🧠 COGNITIVE:
   Consciousness: 0.514351705318464
   Load: 1.20
   Adaptation Rate: 0.100
   Success Rate: 66.6%

   🎯 GOALS:
      • survive: 0.67/1.00 (67%)
      • learn: 0.48/0.80 (60%)
      • adapt: 0.00/0.70 (0%)
      • emerge: 1.00/0.60 (167%)

💾 MEMORY:
   Total Items: 48
   Hot Cache: 17
   Compressed: 31

🕸️  GRAPH:
   Real Nodes: 53
   Imagination: 0
   Total: 53

📡 COMMUNICATION:
   Total Messages: 1,147

END OF REPORT
────────────────────────────────────────────────────────────────────────────────

🟢 C:296 | Mind:0.51